# Penginapan Training: Decision-Driven Data Preparation dan Audit

Notebook ini mendokumentasikan proses persiapan dan audit dataset penginapan Bandung Raya sebagai rangkaian keputusan data engineering.

Pola wajib setiap tahap:

`tujuan kecil -> code/output -> keputusan -> langkah berikutnya`

Notebook ini menggunakan snapshot data terkontrol pada **2026-06-04**. Raw master tidak dibangun ulang otomatis dari JSON karena saat ini sudah mengandung koreksi manual yang dapat hilang jika proses flatten dijalankan tanpa correction layer.

Ruang lingkup:

1. Memastikan sumber dan artefak tersedia.
2. Memeriksa indikasi duplikasi.
3. Memeriksa kandidat duplikat kuat.
4. Memeriksa room-level listing.
5. Menjalankan dan mengevaluasi dedupe level 1.
6. Mengevaluasi kemungkinan duplikat yang tersisa.
7. Mengevaluasi region validation.
8. Menjalankan automated audit.
9. Menentukan temuan actionable tanpa memakai label prioritas angka sebagai struktur keputusan.
10. Memastikan output final tersedia.

Status dataset pada tahap ini adalah **canonical candidate**, belum production canonical final.


## Tahap 0 - Menetapkan Environment dan Artefak

**Tujuan kecil:** memastikan notebook dijalankan dari project yang benar dan seluruh file wajib tersedia sebelum audit dimulai.

In [1]:
from pathlib import Path
import json
import subprocess
import sys

import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT.name != "PIJAK" and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent

WORKSPACE = PROJECT_ROOT / "Penginapan_Workspace"
RAW_DIR = WORKSPACE / "01_Raw_Data" / "generated_raw_csv"
CURATED_DIR = WORKSPACE / "02_Curated"
SCRIPT_DIR = WORKSPACE / "06_Scripts"

paths = {
    "raw_master": RAW_DIR / "HOTEL_GOOGLE_SEARCH_RAW_MASTER_2026-06-02.csv",
    "source_inventory": RAW_DIR / "HOTEL_GOOGLE_SEARCH_JSON_SOURCE_INVENTORY_2026-06-02.csv",
    "existing_canonical": CURATED_DIR / "HOTEL_CANONICAL_CIMAHI_2026-05-29.csv",
    "candidate": CURATED_DIR / "PENGINAPAN_CANONICAL_CANDIDATE_BANDUNG_RAYA_2026-06-02.csv",
    "deduped": CURATED_DIR / "PENGINAPAN_DEDUPED_MASTER_BANDUNG_RAYA_2026-06-02.csv",
    "raw_summary": CURATED_DIR / "hotel_google_search_raw_master_summary_2026-06-02.json",
    "candidate_summary": CURATED_DIR / "penginapan_canonical_candidate_summary_2026-06-02.json",
    "candidate_validation": CURATED_DIR / "penginapan_canonical_candidate_validation_2026-06-02.json",
    "audit_summary": CURATED_DIR / "penginapan_canonical_candidate_automated_audit_2026-06-02.json",
    "findings": CURATED_DIR / "PENGINAPAN_CANONICAL_CANDIDATE_AUTOMATED_AUDIT_FINDINGS_2026-06-02.csv",
    "manual_queue": CURATED_DIR / "PENGINAPAN_MANUAL_REVIEW_QUEUE_AUTOMATED_AUDIT_2026-06-02.csv",
    "adjustment_queue": CURATED_DIR / "PENGINAPAN_AUTOMATED_AUDIT_ADJUSTMENT_PRIORITY_2026-06-02.csv",
}

environment_check = pd.DataFrame(
    [
        {
            "artifact": name,
            "exists": path.exists(),
            "size_kb": round(path.stat().st_size / 1024, 1) if path.exists() else None,
            "modified": path.stat().st_mtime if path.exists() else None,
        }
        for name, path in paths.items()
    ]
)

print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"Python: {sys.executable}")
display(environment_check)

PROJECT_ROOT: D:\File\file\Fauzan Lubada\PIJAK
Python: D:\File\file\Fauzan Lubada\PIJAK\.venv_clean_verify\Scripts\python.exe


,artifact,exists,size_kb,modified
0,raw_master,True,13757.3,1.780404e+09
1,source_inventory,True,5.1,1.780389e+09
2,existing_canonical,True,156.4,1.780036e+09
3,candidate,True,1705.0,1.780628e+09
4,deduped,True,8087.6,1.780628e+09
5,raw_summary,True,2.2,1.780389e+09
6,candidate_summary,True,1.6,1.780628e+09
7,candidate_validation,True,0.3,1.780628e+09
8,audit_summary,True,2.3,1.780628e+09
9,findings,True,710.0,1.780628e+09


### Keputusan: Environment layak digunakan

**Bukti dari output sebelumnya**

- Seluruh artefak wajib tersedia pada workspace penginapan.
- Interpreter Python dan project root dapat ditemukan.

**Keputusan snapshot 2026-06-04**

Audit dapat dilanjutkan menggunakan artefak saat ini. Raw master diperlakukan sebagai snapshot terkontrol dan tidak di-flatten ulang otomatis.

**Langkah berikutnya**

Audit sumber JSON dan inventory untuk memastikan lineage data dapat ditelusuri.

> Jika output cell sebelumnya berubah saat notebook dijalankan ulang, keputusan ini wajib ditinjau ulang sebelum proses dilanjutkan.

## Tahap 1 - Audit Sumber Data

**Tujuan kecil:** menghitung jumlah sumber JSON, sumber yang diproses, file duplikat penuh, jumlah record, dan coverage query.

In [2]:
raw_summary = json.loads(paths["raw_summary"].read_text(encoding="utf-8"))
inventory_df = pd.read_csv(paths["source_inventory"], dtype=str, keep_default_na=False)

source_metrics = pd.DataFrame(
    {
        "metric": [
            "JSON terdeteksi",
            "JSON diproses",
            "File duplikat penuh",
            "Raw records",
            "Unique names raw",
            "Unique property tokens",
        ],
        "value": [
            raw_summary["json_source_count"],
            raw_summary["json_processed_count"],
            raw_summary["json_duplicate_file_count"],
            raw_summary["raw_records"],
            raw_summary["raw_unique_names"],
            raw_summary["raw_unique_property_tokens"],
        ],
    }
)

display(source_metrics)
display(
    inventory_df[
        ["source_file", "duplicate_status", "processed", "query", "records_total", "unique_names_raw"]
    ].sort_values(["processed", "source_file"], ascending=[False, True])
)

,metric,value
0,JSON terdeteksi,18
1,JSON diproses,17
2,File duplikat penuh,1
3,Raw records,2840
4,Unique names raw,1377
5,Unique property tokens,1519


,source_file,duplicate_status,processed,query,records_total,unique_names_raw
17,Salinan dataset_google-hotels-search-scraper_2...,unique_file,True,"hotels in West Bandung Regency, West Java, Ind...",209,163
0,dataset_google-hotels-search-scraper_2026-06-0...,unique_file,True,hotels in Kota Bandung,609,432
1,dataset_google-hotels-search-scraper_2026-06-0...,unique_file,True,"penginapan in Kabupaten Bandung, Jawa Barat",511,419
2,dataset_google-hotels-search-scraper_2026-06-0...,unique_file,True,"penginapan in Cililin, Kabupaten Bandung Barat...",151,120
3,dataset_google-hotels-search-scraper_2026-06-0...,unique_file,True,"penginapan in Saguling, Kabupaten Bandung Bara...",82,67
4,dataset_google-hotels-search-scraper_2026-06-0...,unique_file,True,"penginapan in Cipongkor, Kabupaten Bandung Bar...",83,68
5,dataset_google-hotels-search-scraper_2026-06-0...,unique_file,True,"penginapan in Gununghalu, Kabupaten Bandung Ba...",115,87
6,dataset_google-hotels-search-scraper_2026-06-0...,unique_file,True,"penginapan in Rongga, Kabupaten Bandung Barat,...",211,162
7,dataset_google-hotels-search-scraper_2026-06-0...,unique_file,True,"penginapan in Sindangkerta, Kabupaten Bandung ...",118,91
8,dataset_google-hotels-search-scraper_2026-06-0...,unique_file,True,"penginapan in Padalarang, Kabupaten Bandung Ba...",83,68


### Keputusan: Sumber data diterima sebagai input audit

**Bukti dari output sebelumnya**

- JSON terdeteksi: **18**.
- JSON diproses: **17**.
- File duplikat penuh yang dilewati: **1**.
- Raw records: **2840**.

**Keputusan snapshot 2026-06-04**

Lineage sumber cukup jelas untuk melanjutkan audit. File duplikat penuh tidak perlu diproses kembali karena hanya menggandakan data yang sama.

**Langkah berikutnya**

Periksa raw master untuk melihat apakah duplikasi antar hasil pencarian masih signifikan.

> Jika output cell sebelumnya berubah saat notebook dijalankan ulang, keputusan ini wajib ditinjau ulang sebelum proses dilanjutkan.

## Tahap 2 - Audit Raw Master dan Indikasi Duplikasi

**Tujuan kecil:** membandingkan jumlah row, nama unik, property token, sumber query, dan kelengkapan dasar sebelum dedupe.

In [3]:
raw_df = pd.read_csv(paths["raw_master"], dtype=str, keep_default_na=False)
existing_df = pd.read_csv(paths["existing_canonical"], dtype=str, keep_default_na=False)

raw_audit = pd.DataFrame(
    {
        "metric": [
            "Raw master rows",
            "Existing canonical rows",
            "Raw unique names",
            "Raw unique property tokens",
            "Rows dengan koordinat",
            "Rows dengan rating",
            "Rows dengan review count",
        ],
        "value": [
            len(raw_df),
            len(existing_df),
            raw_df["name"].nunique(dropna=True),
            raw_df["property_token"].replace("", pd.NA).nunique(dropna=True),
            int(((raw_df["latitude"] != "") & (raw_df["longitude"] != "")).sum()),
            int((raw_df["overall_rating"] != "").sum()),
            int((raw_df["reviews"] != "").sum()),
        ],
    }
)

display(raw_audit)
display(raw_df["source_query_area"].value_counts().rename_axis("query_area").reset_index(name="rows"))

,metric,value
0,Raw master rows,2840
1,Existing canonical rows,181
2,Raw unique names,1383
3,Raw unique property tokens,1519
4,Rows dengan koordinat,2840
5,Rows dengan rating,1528
6,Rows dengan review count,1528


,query_area,rows
0,Kota Bandung,609
1,Kabupaten Bandung,511
2,"Rongga, Kabupaten Bandung Barat",211
3,West Bandung Regency,209
4,"Cikalongwetan, Kabupaten Bandung Barat",186
5,"Cipeundeuy, Kabupaten Bandung Barat",164
6,"Cililin, Kabupaten Bandung Barat",151
7,"Sindangkerta, Kabupaten Bandung Barat",118
8,"Gununghalu, Kabupaten Bandung Barat",115
9,"Pangalengan, Kabupaten Bandung",106


### Keputusan: Dedupe level 1 diperlukan

**Bukti dari output sebelumnya**

- Raw master berisi **2840** rows, tetapi hanya **1377** nama unik.
- Pool gabungan yang akan diperiksa berisi **3021** rows.

**Keputusan snapshot 2026-06-04**

Perbedaan besar antara jumlah rows dan nama unik menunjukkan banyak listing berulang antar query/page. Namun kesamaan nama saja belum cukup aman untuk menggabungkan data.

**Langkah berikutnya**

Bangun kandidat duplikat kuat menggunakan dedupe key konservatif dan ukur bukti penggabungannya.

> Jika output cell sebelumnya berubah saat notebook dijalankan ulang, keputusan ini wajib ditinjau ulang sebelum proses dilanjutkan.

## Tahap 3 - Kandidat Duplikat Kuat

**Tujuan kecil:** mengetahui berapa kelompok yang benar-benar memenuhi aturan dedupe level 1 dan sinyal apa yang digunakan.

Aturan dedupe level 1:

1. Prioritas pertama: `property_token` yang sama.
2. Jika token tidak tersedia: nama ternormalisasi dan koordinat rounded yang sama.
3. Jika koordinat tidak tersedia: hanya nama ternormalisasi yang benar-benar sama.
4. Pemilihan record utama mempertimbangkan tipe properti, sumber, kualitas data, reviews, rating, dan page.

In [4]:
sys.path.insert(0, str(SCRIPT_DIR))
import build_penginapan_canonical_candidate as dedupe_pipeline

raw_for_pool = dedupe_pipeline.load_raw_master(paths["raw_master"])
existing_for_pool = dedupe_pipeline.load_existing_canonical(paths["existing_canonical"])
pool_df = dedupe_pipeline.prepare_pool(raw_for_pool, existing_for_pool)

group_sizes = pool_df.groupby("dedupe_key").size().sort_values(ascending=False)
duplicate_groups = group_sizes[group_sizes > 1]
duplicate_key_type = duplicate_groups.index.to_series().str.split("::").str[0].value_counts()

strong_duplicate_metrics = pd.DataFrame(
    {
        "metric": ["Pool rows", "Unique dedupe keys", "Duplicate groups", "Rows in duplicate groups", "Rows removable"],
        "value": [
            len(pool_df),
            group_sizes.size,
            len(duplicate_groups),
            int(duplicate_groups.sum()),
            int((duplicate_groups - 1).sum()),
        ],
    }
)

display(strong_duplicate_metrics)
display(duplicate_key_type.rename_axis("dedupe_key_type").reset_index(name="duplicate_groups"))
display(
    pool_df[pool_df["dedupe_key"].isin(duplicate_groups.head(10).index)][
        ["dedupe_key", "name", "property_type", "latitude", "longitude", "source_query_area", "data_quality_score"]
    ].sort_values(["dedupe_key", "name"])
)

,metric,value
0,Pool rows,3021
1,Unique dedupe keys,1656
2,Duplicate groups,669
3,Rows in duplicate groups,2034
4,Rows removable,1365


,dedupe_key_type,duplicate_groups
0,token,669


,dedupe_key,name,property_type,latitude,longitude,source_query_area,data_quality_score
1370,token::ChkQ-ZKzgpWSq-8XGg0vZy8xMW5ibXE1anhjEAI,Villa Camp Forest by Little Bnb - Three-Bedroo...,villa,-6.8641,107.4574,"Cipongkor, Kabupaten Bandung Barat",0.8
1425,token::ChkQ-ZKzgpWSq-8XGg0vZy8xMW5ibXE1anhjEAI,Villa Camp Forest by Little Bnb - Three-Bedroo...,villa,-6.8641,107.4574,"Cipongkor, Kabupaten Bandung Barat",0.8
1893,token::ChkQ-ZKzgpWSq-8XGg0vZy8xMW5ibXE1anhjEAI,Villa Camp Forest by Little Bnb - Three-Bedroo...,villa,-6.8641,107.4574,"Padalarang, Kabupaten Bandung Barat",0.8
1980,token::ChkQ-ZKzgpWSq-8XGg0vZy8xMW5ibXE1anhjEAI,Villa Camp Forest by Little Bnb - Three-Bedroo...,villa,-6.8641,107.4574,"Batujajar, Kabupaten Bandung Barat",0.8
2026,token::ChkQ-ZKzgpWSq-8XGg0vZy8xMW5ibXE1anhjEAI,Villa Camp Forest by Little Bnb - Three-Bedroo...,villa,-6.8641,107.4574,"Cipatat, Kabupaten Bandung Barat",0.8
...,...,...,...,...,...,...,...
2002,token::ChoQrLmD88qDo_OxARoNL2cvMTFuYm1ybDkyahAC,Cozy house @Tatar simakirana Kota baru Parahya...,villa,-6.8698,107.4557,"Batujajar, Kabupaten Bandung Barat",0.65
2078,token::ChoQrLmD88qDo_OxARoNL2cvMTFuYm1ybDkyahAC,Cozy house @Tatar simakirana Kota baru Parahya...,villa,-6.8698,107.4557,"Cipatat, Kabupaten Bandung Barat",0.65
2188,token::ChoQrLmD88qDo_OxARoNL2cvMTFuYm1ybDkyahAC,Cozy house @Tatar simakirana Kota baru Parahya...,villa,-6.8698,107.4557,"Cipeundeuy, Kabupaten Bandung Barat",0.65
2190,token::ChoQrLmD88qDo_OxARoNL2cvMTFuYm1ybDkyahAC,Cozy house @Tatar simakirana Kota baru Parahya...,villa,-6.8698,107.4557,"Cipeundeuy, Kabupaten Bandung Barat",0.65


### Keputusan: Aturan penggabungan dedupe level 1 diterima

**Bukti dari output sebelumnya**

- Ditemukan **669** kelompok duplikat kuat.
- Kelompok tersebut mencakup **2034** rows.
- Pengurangan potensial: **1365** rows.
- Tipe key kelompok aktif: **{'token': 669}**.

**Keputusan snapshot 2026-06-04**

Pada snapshot ini, seluruh kelompok yang benar-benar digabung berasal dari `property_token` yang sama. Ini merupakan sinyal kuat, sehingga dedupe level 1 cukup aman untuk dijalankan. Aturan fallback tetap konservatif, tetapi tidak menjadi penyebab penggabungan aktif pada snapshot ini.

**Langkah berikutnya**

Periksa room-level listing sebelum menjalankan dedupe agar unit/kamar tidak salah diperlakukan sebagai properti utama.

> Jika output cell sebelumnya berubah saat notebook dijalankan ulang, keputusan ini wajib ditinjau ulang sebelum proses dilanjutkan.

## Tahap 4 - Audit Room-Level Listing

**Tujuan kecil:** mengetahui listing yang merepresentasikan kamar/unit, bukan properti utama.

In [5]:
room_pool_df = pool_df[pool_df["property_type"].eq("room_level_listing")].copy()

print(f"Room-level listing pada pool: {len(room_pool_df)}")
display(
    room_pool_df[
        ["name", "normalized_name", "latitude", "longitude", "source_query_area", "property_token"]
    ].head(30)
)

Room-level listing pada pool: 104


,name,normalized_name,latitude,longitude,source_query_area,property_token
247,RedDoorz Plus near Asia Afrika 3 - Deluxe Room,reddoorz plus asia afrika 3,-6.9237,107.6098,Kota Bandung,ChkQm--YstKS0vxoGg0vZy8xMW5ibXFkNWpwEAI
362,ZEN Rooms Talaga Bodas,zen rooms talaga bodas,-6.9291,107.6244,Kota Bandung,ChoIw9m7nLSkioXNARoNL2cvMTFjczFzdHZzdxAB
370,Little Mykonos - Standard Queen Room,little mykonos,-6.9111,107.6059,Kota Bandung,ChkQ28X7gK3PpMJCGg0vZy8xMXo2YmNrZGxxEAI
382,Urbanview Darmo Residence Bandung - Deluxe Dou...,urbanview darmo residence bandung,-6.8967,107.5875,Kota Bandung,ChoQw4b0mv7EkcXwARoNL2cvMTF6NHNrMTF2MxAC
392,OYO 2088 Grha Blue Sky Syariah - Standard Doub...,oyo 2088 grha blue sky syariah,-6.8843,107.5781,Kota Bandung,ChkQhKiaj-KJ0ugnGg0vZy8xMXo1cV9wX213EAI
409,Little Mykonos - Standard Queen Room,little mykonos,-6.9111,107.6059,Kota Bandung,ChkQ28X7gK3PpMJCGg0vZy8xMXo2YmNrZGxxEAI
410,Urbanview Darmo Residence Bandung - Deluxe Dou...,urbanview darmo residence bandung,-6.8967,107.5875,Kota Bandung,ChoQw4b0mv7EkcXwARoNL2cvMTF6NHNrMTF2MxAC
429,Bantal Guling Trans - Family Triple Room,bantal guling trans,-6.9302,107.6395,Kota Bandung,ChoQ7dXNoPTaud76ARoNL2cvMTF6Mzl0cXN2dBAC
497,Zen Rooms Pangaran Dalem Kaum,zen rooms pangaran dalem kaum,-6.9242,107.6099,Kota Bandung,ChoIh_eL0uvt0YfXARoNL2cvMTFieGZ5eTliMRAB
540,ZEN Rooms Aria Jipang,zen rooms aria jipang,-6.9,107.6162,Kota Bandung,ChkI4fX3xOeJuM0dGg0vZy8xMWRkeHR4c2xzEAE


### Keputusan: Room-level listing dipertahankan tetapi bukan kandidat ranking utama

**Bukti dari output sebelumnya**

- Room-level listing pada pool: **104**.
- Setelah dedupe level 1 diperkirakan tersisa **54**.

**Keputusan snapshot 2026-06-04**

Room-level listing tidak boleh dihapus otomatis karena dapat menyimpan informasi tipe kamar, unit, harga, atau gambar. Listing ini juga tidak boleh bersaing setara dengan properti utama.

**Langkah berikutnya**

Jalankan dedupe level 1 tanpa menghapus room-level listing. Siapkan parent-child mapping pada tahap terpisah.

> Jika output cell sebelumnya berubah saat notebook dijalankan ulang, keputusan ini wajib ditinjau ulang sebelum proses dilanjutkan.

## Tahap 5 - Menjalankan Dedupe Konservatif Level 1

**Tujuan kecil:** membangun ulang deduped master dan canonical candidate menggunakan aturan dedupe yang sudah diterima.

Catatan: proses ini membaca raw master terkontrol yang sudah tersedia. Proses tidak membangun ulang raw master dari JSON.

In [6]:
dedupe_run = subprocess.run(
    [
        sys.executable,
        str(SCRIPT_DIR / "build_penginapan_canonical_candidate.py"),
        "--skip-notebook-update",
    ],
    check=True,
    capture_output=True,
    text=True,
)
print(dedupe_run.stdout.strip())

candidate_summary = json.loads(paths["candidate_summary"].read_text(encoding="utf-8"))
candidate_validation = json.loads(paths["candidate_validation"].read_text(encoding="utf-8"))

display(pd.DataFrame({
    "metric": ["Pool rows", "Candidate rows", "Rows removed", "Validation gate"],
    "value": [
        candidate_summary["pool_rows"],
        candidate_summary["candidate_rows"],
        candidate_summary["dedupe_removed_rows"],
        candidate_validation["gate"],
    ],
}))

pool_rows=3021
candidate_rows=1656
dedupe_removed_rows=1365
validation_gate=PASS_WITH_WARNINGS
candidate_output=D:\File\file\Fauzan Lubada\PIJAK\Penginapan_Workspace\02_Curated\PENGINAPAN_CANONICAL_CANDIDATE_BANDUNG_RAYA_2026-06-02.csv


,metric,value
0,Pool rows,3021
1,Candidate rows,1656
2,Rows removed,1365
3,Validation gate,PASS_WITH_WARNINGS


### Keputusan: Dedupe level 1 diterima dengan peringatan

**Bukti dari output sebelumnya**

- Pool: **3021** rows.
- Candidate: **1656** rows.
- Rows yang direduksi: **1365**.

**Keputusan snapshot 2026-06-04**

Hasil dedupe level 1 diterima karena penggabungan aktif didukung property token yang sama. Status masih `canonical candidate`, bukan final, karena kemungkinan parent-child dan duplikat semantik masih perlu ditangani.

**Langkah berikutnya**

Audit pola duplikasi yang masih tersisa setelah dedupe level 1.

> Jika output cell sebelumnya berubah saat notebook dijalankan ulang, keputusan ini wajib ditinjau ulang sebelum proses dilanjutkan.

## Tahap 6 - Audit Kemungkinan Duplikat yang Tersisa

**Tujuan kecil:** mencari kandidat dengan nama ternormalisasi dan koordinat sama yang masih tersisa setelah dedupe level 1.

Kondisi ini tidak otomatis berarti duplikat. Kelompok dapat berisi properti utama, kamar, unit apartment, atau listing berbeda pada lokasi yang sama.

In [7]:
candidate_df = pd.read_csv(paths["candidate"], dtype=str, keep_default_na=False)
possible_key = (
    candidate_df["normalized_name"].astype(str)
    + "|"
    + candidate_df["latitude"].astype(str)
    + "|"
    + candidate_df["longitude"].astype(str)
)
possible_mask = possible_key.duplicated(keep=False)
possible_df = candidate_df[possible_mask].copy()
possible_df["possible_duplicate_key"] = possible_key[possible_mask]

remaining_metrics = pd.DataFrame(
    {
        "metric": ["Candidate rows", "Possible duplicate rows", "Possible duplicate groups"],
        "value": [len(candidate_df), len(possible_df), possible_df["possible_duplicate_key"].nunique()],
    }
)
display(remaining_metrics)
display(
    possible_df[
        ["possible_duplicate_key", "penginapan_id", "name", "property_type", "overall_rating", "reviews"]
    ].sort_values(["possible_duplicate_key", "property_type", "name"]).head(40)
)

,metric,value
0,Candidate rows,1656
1,Possible duplicate rows,233
2,Possible duplicate groups,104


,possible_duplicate_key,penginapan_id,name,property_type,overall_rating,reviews
1336,alesha property apartment|-6.9073|107.6451,LODGE-01337,Alesha Property - Studio Apartment,apartment,,
1413,alesha property apartment|-6.9073|107.6451,LODGE-01414,Alesha Property - Studio Apartment,apartment,,
852,artotel suites aquila bandung|-6.8954|107.5893,LODGE-00853,ARTOTEL Suites Aquila Bandung,hotel,4.4,2172
1605,artotel suites aquila bandung|-6.8954|107.5893,LODGE-01606,ARTOTEL Suites Aquila Bandung,hotel,4.4,2170
524,aubrey villa ciwidey chalet|-7.124|107.5054,LODGE-00525,Aubrey Villa Ciwidey - Chalet,villa,,
557,aubrey villa ciwidey chalet|-7.124|107.5054,LODGE-00558,Aubrey Villa Ciwidey - Chalet,villa,,
558,aubrey villa ciwidey chalet|-7.124|107.5054,LODGE-00559,Aubrey Villa Ciwidey - Chalet,villa,,
1112,bantal guling trans|-6.9302|107.6395,LODGE-01113,Bantal Guling Trans - Family Triple Room,room_level_listing,2.5,8
873,bantal guling trans|-6.9302|107.6395,LODGE-00874,Bantal Guling Trans - Standard Family Room,room_level_listing,2.5,8
874,bantal guling trans|-6.9302|107.6395,LODGE-00875,Bantal Guling Trans - Superior Family Room,room_level_listing,2.5,8


### Keputusan: Duplikat tersisa tidak boleh digabung otomatis

**Bukti dari output sebelumnya**

- Terdapat **104** kelompok atau **233** rows dengan nama ternormalisasi dan koordinat yang sama.

**Keputusan snapshot 2026-06-04**

Sinyal nama dan koordinat yang sama belum cukup untuk auto-merge karena dapat merepresentasikan parent-child, tipe kamar, atau unit berbeda. Kasus ini harus masuk review lanjutan dan parent-child preparation.

**Langkah berikutnya**

Lanjutkan region validation tanpa menghapus kelompok tersisa.

> Jika output cell sebelumnya berubah saat notebook dijalankan ulang, keputusan ini wajib ditinjau ulang sebelum proses dilanjutkan.


## Tahap 7 - Region Validation

**Tujuan kecil:** memastikan kandidat berada dalam cakupan rough Bandung Raya dan mengidentifikasi border/outside/needs-review.

Region validation saat ini berbasis bounding coordinate dan bucket kasar. Hasilnya tidak boleh dianggap sebagai batas administratif presisi.

In [8]:
region_counts = candidate_df["region_validation_label"].value_counts(dropna=False)
bucket_counts = candidate_df["region_bucket"].value_counts(dropna=False)
region_exceptions = candidate_df[~candidate_df["region_validation_label"].eq("bandung_raya_valid")]

display(region_counts.rename_axis("region_validation_label").reset_index(name="rows"))
display(bucket_counts.rename_axis("region_bucket").reset_index(name="rows"))
display(
    region_exceptions[
        ["penginapan_id", "name", "latitude", "longitude", "region_bucket", "region_validation_label", "region_validation_reason"]
    ]
)

,region_validation_label,rows
0,bandung_raya_valid,1656


,region_bucket,rows
0,kota_bandung,805
1,cimahi_baros_pasteur,443
2,kbb_tengah_barat,317
3,kbb_utara_lembang_parongpong_cisarua,34
4,kbb_selatan_baratdaya,28
5,bandung_raya_other,27
6,kabupaten_bandung,2


,penginapan_id,name,latitude,longitude,region_bucket,region_validation_label,region_validation_reason


### Keputusan: Region validation diterima sebagai filter kasar

**Bukti dari output sebelumnya**

- Kandidat `bandung_raya_valid`: **1656**.
- Kandidat non-valid/needs-review: **0**.

**Keputusan snapshot 2026-06-04**

Seluruh kandidat saat ini lolos rough Bandung Raya. Hasil ini cukup untuk filter awal rekomendasi, tetapi belum cukup untuk klaim wilayah administratif yang presisi.

**Langkah berikutnya**

Jalankan automated audit untuk schema, ID, numeric range, missing data, room-level listing, dan possible duplicate.

> Jika output cell sebelumnya berubah saat notebook dijalankan ulang, keputusan ini wajib ditinjau ulang sebelum proses dilanjutkan.

## Tahap 8 - Menjalankan Automated Audit

**Tujuan kecil:** menghasilkan gate dan daftar temuan tanpa mengubah data candidate.

Audit mencakup schema, ID, koordinat, region, rating, reviews, harga, quality score, property type, room-level listing, possible duplicate, dan missing field penting.

In [9]:
audit_run = subprocess.run(
    [
        sys.executable,
        str(SCRIPT_DIR / "audit_penginapan_canonical_candidate.py"),
        "--skip-notebook-update",
    ],
    check=True,
    capture_output=True,
    text=True,
)
print(audit_run.stdout.strip())

audit_summary = json.loads(paths["audit_summary"].read_text(encoding="utf-8"))
findings_df = pd.read_csv(paths["findings"], dtype=str, keep_default_na=False)

display(pd.DataFrame({
    "metric": ["Gate", "Rows", "Findings", "Affected rows", "ERROR findings"],
    "value": [
        audit_summary["gate"],
        audit_summary["row_count"],
        audit_summary["finding_count"],
        audit_summary["affected_row_count"],
        int((findings_df["severity"] == "ERROR").sum()),
    ],
}))
display(findings_df["code"].value_counts().rename_axis("finding_code").reset_index(name="count"))

gate=PASS_WITH_WARNINGS
rows=1656
findings=2944
affected_rows=1274
severity_counts={'MANUAL_REVIEW': 2474, 'WARNING': 470}
findings_csv=D:\File\file\Fauzan Lubada\PIJAK\Penginapan_Workspace\02_Curated\PENGINAPAN_CANONICAL_CANDIDATE_AUTOMATED_AUDIT_FINDINGS_2026-06-02.csv
manual_review_queue=D:\File\file\Fauzan Lubada\PIJAK\Penginapan_Workspace\02_Curated\PENGINAPAN_MANUAL_REVIEW_QUEUE_AUTOMATED_AUDIT_2026-06-02.csv
adjustment_priority_queue=D:\File\file\Fauzan Lubada\PIJAK\Penginapan_Workspace\02_Curated\PENGINAPAN_AUTOMATED_AUDIT_ADJUSTMENT_PRIORITY_2026-06-02.csv


,metric,value
0,Gate,PASS_WITH_WARNINGS
1,Rows,1656
2,Findings,2944
3,Affected rows,1274
4,ERROR findings,0


,finding_code,count
0,OVERALL_RATING_MISSING,666
1,REVIEWS_MISSING,666
2,PRICE_LOWEST_MISSING,637
3,LINK_MISSING,470
4,POSSIBLE_DUPLICATE_NAME_COORD,247
5,AMENITIES_MISSING,154
6,ROOM_LEVEL_LISTING,54
7,PRIMARY_IMAGE_URL_MISSING,50


### Keputusan: Candidate dapat dilanjutkan dengan caveat, bukan dianggap final

**Bukti dari output sebelumnya**

- Audit gate: **PASS_WITH_WARNINGS**.
- Temuan: **2944** pada **1274** rows.
- ERROR findings: **0**.

**Keputusan snapshot 2026-06-04**

Tidak ada error yang memblokir dataset. Banyak temuan berasal dari field opsional yang kosong dan kasus manual review. Dataset boleh dipakai untuk pengembangan lanjutan jika sistem membaca quality flags dan tidak membuat klaim atas data yang kosong.

**Langkah berikutnya**

Pisahkan temuan actionable agar pekerjaan berikutnya terarah.

> Jika output cell sebelumnya berubah saat notebook dijalankan ulang, keputusan ini wajib ditinjau ulang sebelum proses dilanjutkan.


## Tahap 9 - Ringkasan Temuan Actionable

**Tujuan kecil:** membedakan temuan yang memblokir penggunaan data dari temuan yang masih bisa dilanjutkan dengan review atau caveat.

Istilah yang dipakai pada notebook ini:

- **ERROR aktif:** harus diperbaiki sebelum dataset dipakai.
- **Review utama:** penting untuk struktur ranking, duplicate review, dan parent-child.
- **Caveat:** data boleh dipakai, tetapi sistem tidak boleh mengklaim field yang kosong.


In [10]:
adjustment_df = pd.read_csv(paths["adjustment_queue"], dtype=str, keep_default_na=False)
manual_queue_df = pd.read_csv(paths["manual_queue"], dtype=str, keep_default_na=False)

actionable_summary = pd.DataFrame(
    {
        "metric": [
            "Manual review queue rows",
            "Actionable queue rows",
            "Rows terkait possible duplicate",
            "Rows terkait room-level listing",
        ],
        "value": [
            len(manual_queue_df),
            len(adjustment_df),
            int(adjustment_df["finding_codes"].str.contains("POSSIBLE_DUPLICATE_NAME_COORD", na=False).sum()),
            int(adjustment_df["finding_codes"].str.contains("ROOM_LEVEL_LISTING", na=False).sum()),
        ],
    }
)

display(actionable_summary)
display(adjustment_df[["penginapan_id", "name", "finding_codes", "recommended_next_step"]].head(40))


,metric,value
0,Manual review queue rows,301
1,Actionable queue rows,278
2,Rows terkait possible duplicate,247
3,Rows terkait room-level listing,54


,penginapan_id,name,finding_codes,recommended_next_step
0,LODGE-00087,Budi House Food Station - Deluxe Twin Room,POSSIBLE_DUPLICATE_NAME_COORD; ROOM_LEVEL_LISTING,Review duplicate candidate; gabungkan jika mem...
1,LODGE-00088,Budi House Food Station - Standard Double Room,POSSIBLE_DUPLICATE_NAME_COORD; ROOM_LEVEL_LISTING,Review duplicate candidate; gabungkan jika mem...
2,LODGE-00120,Sans Stay Skyland Pasteur Bandung - Deluxe Dou...,POSSIBLE_DUPLICATE_NAME_COORD; ROOM_LEVEL_LISTING,Review duplicate candidate; gabungkan jika mem...
3,LODGE-00121,Sans Stay Skyland Pasteur Bandung - Standard D...,POSSIBLE_DUPLICATE_NAME_COORD; ROOM_LEVEL_LISTING,Review duplicate candidate; gabungkan jika mem...
4,LODGE-00150,Budi House Food Station - Superior Double Room,POSSIBLE_DUPLICATE_NAME_COORD; ROOM_LEVEL_LISTING,Review duplicate candidate; gabungkan jika mem...
5,LODGE-00239,Little Blue House - Queen Room,POSSIBLE_DUPLICATE_NAME_COORD; ROOM_LEVEL_LISTING,Review duplicate candidate; gabungkan jika mem...
6,LODGE-00488,Penginapan Barkah - Family Room,POSSIBLE_DUPLICATE_NAME_COORD; ROOM_LEVEL_LISTING,Review duplicate candidate; gabungkan jika mem...
7,LODGE-00489,Penginapan Barkah - Family Room,POSSIBLE_DUPLICATE_NAME_COORD; ROOM_LEVEL_LISTING,Review duplicate candidate; gabungkan jika mem...
8,LODGE-00490,Penginapan Barkah - Standard Family Room,POSSIBLE_DUPLICATE_NAME_COORD; ROOM_LEVEL_LISTING,Review duplicate candidate; gabungkan jika mem...
9,LODGE-00510,Pondok Stevia Ciwidey - Deluxe Double Room,POSSIBLE_DUPLICATE_NAME_COORD; ROOM_LEVEL_LISTING,Review duplicate candidate; gabungkan jika mem...


### Keputusan: Fokus berikutnya adalah duplicate review dan parent-child

**Bukti dari output sebelumnya**

- Tidak ada ERROR aktif yang memblokir dataset.
- Temuan actionable masih didominasi possible duplicate dan room-level listing.

**Keputusan snapshot 2026-06-05**

Dataset boleh dilanjutkan ke entity resolution candidate. Temuan duplicate dan room-level tidak diselesaikan dengan penghapusan massal.

**Langkah berikutnya**

Buat file candidate mapping: possible duplicate review, room-level listing, dan parent-child mapping candidate.

> Jika output cell sebelumnya berubah saat notebook dijalankan ulang, keputusan ini wajib ditinjau ulang sebelum proses dilanjutkan.


## Tahap 10 - Verifikasi Output Final Audit

**Tujuan kecil:** memastikan seluruh artefak hasil audit tersedia dan dapat digunakan oleh tahap berikutnya.

In [11]:
final_artifacts = {
    "raw_master": paths["raw_master"],
    "deduped_master": paths["deduped"],
    "canonical_candidate": paths["candidate"],
    "automated_findings": paths["findings"],
    "manual_review_queue": paths["manual_queue"],
    "adjustment_priority_queue": paths["adjustment_queue"],
}

artifact_rows = []
for name, path in final_artifacts.items():
    row_count = None
    column_count = None
    if path.exists() and path.suffix.lower() == ".csv":
        preview = pd.read_csv(path, dtype=str, keep_default_na=False)
        row_count = len(preview)
        column_count = len(preview.columns)
    artifact_rows.append(
        {
            "artifact": name,
            "exists": path.exists(),
            "rows": row_count,
            "columns": column_count,
            "path": str(path),
        }
    )

display(pd.DataFrame(artifact_rows))

,artifact,exists,rows,columns,path
0,raw_master,True,2840,82,D:\File\file\Fauzan Lubada\PIJAK\Penginapan_Wo...
1,deduped_master,True,1656,99,D:\File\file\Fauzan Lubada\PIJAK\Penginapan_Wo...
2,canonical_candidate,True,1656,44,D:\File\file\Fauzan Lubada\PIJAK\Penginapan_Wo...
3,automated_findings,True,2944,7,D:\File\file\Fauzan Lubada\PIJAK\Penginapan_Wo...
4,manual_review_queue,True,301,7,D:\File\file\Fauzan Lubada\PIJAK\Penginapan_Wo...
5,adjustment_priority_queue,True,278,6,D:\File\file\Fauzan Lubada\PIJAK\Penginapan_Wo...


### Keputusan: Audit keseluruhan selesai untuk snapshot saat ini

**Bukti dari output sebelumnya**

- Canonical candidate tersedia dengan **1656** rows.
- Dedupe level 1 mereduksi **1365** rows.
- Audit gate: **PASS_WITH_WARNINGS** dan tidak memiliki ERROR aktif.

**Keputusan snapshot 2026-06-04**

Pondasi data cukup untuk melanjutkan pekerjaan struktur penginapan, tetapi belum menjadi canonical final. Candidate belum boleh dianggap bersih sepenuhnya dari parent-child atau duplikat semantik.

**Langkah berikutnya**

Tahap terbaik selanjutnya adalah merancang parent-child mapping untuk room-level listing dan kelompok kemungkinan duplikat, kemudian membuat target scraping review hanya dari properti utama.

> Jika output cell sebelumnya berubah saat notebook dijalankan ulang, keputusan ini wajib ditinjau ulang sebelum proses dilanjutkan.

## Ringkasan Keputusan Akhir

| Area | Keputusan |
|---|---|
| Sumber data | Diterima; lineage tersedia dan file duplikat penuh dilewati |
| Raw master | Dipakai sebagai snapshot terkontrol; jangan rebuild dari JSON sebelum correction layer tersedia |
| Dedupe level 1 | Diterima; penggabungan aktif didukung property token yang sama |
| Room-level listing | Dipertahankan; jangan diranking setara dengan properti utama |
| Possible duplicate tersisa | Jangan auto-merge; siapkan review dan parent-child |
| Region validation | Diterima sebagai rough filter, bukan batas administratif presisi |
| Automated audit | `PASS_WITH_WARNINGS`; tidak ada ERROR aktif |
| Status dataset | Canonical candidate, belum production canonical final |
| Next step | Parent-child mapping lalu target scraping review properti utama |


## Fase B - Entity Resolution Candidate Penginapan

Fase ini menyiapkan struktur awal parent-child untuk penginapan.

Output fase ini **belum final**. Semua relasi masih berstatus candidate mapping agar bisa direview sebelum dipakai untuk ranking atau target scraping review.

## Tahap A - Kunci Snapshot Data Saat Ini

**Tujuan kecil:** memastikan candidate saat ini menjadi basis kerja dan tidak dibangun ulang dari JSON.

In [12]:
from pathlib import Path
import json
import subprocess
import sys

import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT.name != "PIJAK" and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent

WORKSPACE = PROJECT_ROOT / "Penginapan_Workspace"
CURATED_DIR = WORKSPACE / "02_Curated"
SCRIPT_DIR = WORKSPACE / "06_Scripts"

candidate_path = CURATED_DIR / "PENGINAPAN_CANONICAL_CANDIDATE_BANDUNG_RAYA_2026-06-02.csv"
candidate_df = pd.read_csv(candidate_path, dtype=str, keep_default_na=False)

snapshot_table = pd.DataFrame(
    {
        "metric": [
            "Candidate rows",
            "Property types",
            "Room-level listing rows",
            "Region valid rows",
        ],
        "value": [
            len(candidate_df),
            candidate_df["property_type"].nunique(),
            int(candidate_df["property_type"].eq("room_level_listing").sum()),
            int(candidate_df["region_validation_label"].eq("bandung_raya_valid").sum()),
        ],
    }
)

display(snapshot_table)
display(candidate_df["property_type"].value_counts().rename_axis("property_type").reset_index(name="rows"))

,metric,value
0,Candidate rows,1656
1,Property types,6
2,Room-level listing rows,54
3,Region valid rows,1656


,property_type,rows
0,hotel,487
1,apartment,352
2,villa,342
3,vacation_rental,269
4,guest_house,152
5,room_level_listing,54


### Keputusan Tahap A

Candidate saat ini dipakai sebagai snapshot kerja. Data tidak di-rebuild dari JSON pada tahap ini karena kita sedang menyiapkan struktur entity resolution, bukan mengulang ingest data.

## Tahap B - Audit Possible Duplicate

**Tujuan kecil:** melihat penginapan yang masih berpotensi dobel berdasarkan nama ternormalisasi dan koordinat rounded.

In [13]:
parent_child_run = subprocess.run(
    [
        sys.executable,
        str(SCRIPT_DIR / "build_penginapan_parent_child_candidates.py"),
    ],
    check=True,
    capture_output=True,
    text=True,
)
print(parent_child_run.stdout.strip())

summary_path = CURATED_DIR / "penginapan_parent_child_candidate_summary_2026-06-05.json"
duplicate_review_path = CURATED_DIR / "PENGINAPAN_POSSIBLE_DUPLICATE_REVIEW_2026-06-05.csv"

parent_child_summary = json.loads(summary_path.read_text(encoding="utf-8"))
duplicate_review_df = pd.read_csv(duplicate_review_path, dtype=str, keep_default_na=False)

display(
    pd.DataFrame(
        {
            "metric": ["Possible duplicate rows", "Possible duplicate groups"],
            "value": [
                parent_child_summary["possible_duplicate_rows"],
                parent_child_summary["possible_duplicate_groups"],
            ],
        }
    )
)
display(duplicate_review_df["group_review_type"].value_counts().rename_axis("group_review_type").reset_index(name="rows"))
display(duplicate_review_df.head(30))

candidate_input_rows=1656
possible_duplicate_rows=247
possible_duplicate_groups=111
room_level_listing_rows=54
parent_child_mapping_rows=54
child_with_parent_candidate=13
child_without_parent_candidate=41
mapping_output=D:\File\file\Fauzan Lubada\PIJAK\Penginapan_Workspace\02_Curated\PENGINAPAN_PARENT_CHILD_MAPPING_CANDIDATE_2026-06-05.csv


,metric,value
0,Possible duplicate rows,247
1,Possible duplicate groups,111


,group_review_type,rows
0,same_type_duplicate_candidate,218
1,room_variant_group,18
2,parent_child_candidate,9
3,mixed_type_duplicate_candidate,2


,possible_duplicate_group_id,possible_duplicate_key,group_size,group_property_types,group_review_type,recommended_decision,penginapan_id,name,normalized_name,property_type,latitude,longitude,overall_rating,reviews,price_lowest,data_quality_score,data_quality_label,link,primary_image_url,dedupe_group_size
0,DUP-20260605-0001,alesha property apartment|-6.9073|107.6451,2,apartment,same_type_duplicate_candidate,Review duplikat semantik; gabungkan hanya jika...,LODGE-01337,Alesha Property - Studio Apartment,alesha property apartment,apartment,-6.9073,107.6451,,,269248,0.85,high,https://www.bluepillow.com/search/68123ee55250...,https://q-xx.bstatic.com/xdata/images/hotel/ma...,2
1,DUP-20260605-0001,alesha property apartment|-6.9073|107.6451,2,apartment,same_type_duplicate_candidate,Review duplikat semantik; gabungkan hanya jika...,LODGE-01414,Alesha Property - Studio Apartment,alesha property apartment,apartment,-6.9073,107.6451,,,269248,0.8,high,https://www.freecancellations.com/search/F2d3Q...,https://freecancellationscdn-hsbgcwfragasdpfe....,2
2,DUP-20260605-0002,apartemen the edge baros kamarku|-6.8979|107.5368,2,apartment,same_type_duplicate_candidate,Review duplikat semantik; gabungkan hanya jika...,LODGE-00101,Apartemen The Edge Baros - Kamarku,apartemen the edge baros kamarku,apartment,-6.8979256,107.536754,3.6,182,168399,0.88,high,https://www.reddoorz.com/id-id/hotel/indonesia...,https://images.trvl-media.com/lodging/96000000...,1
3,DUP-20260605-0002,apartemen the edge baros kamarku|-6.8979|107.5368,2,apartment,same_type_duplicate_candidate,Review duplikat semantik; gabungkan hanya jika...,LODGE-00350,Apartemen The Edge Baros - Kamarku,apartemen the edge baros kamarku,apartment,-6.8979,107.5368,3.7,188,,0.6,medium,https://www.google.com/aclk?sa=l&ai=DChsSEwi45...,,1
4,DUP-20260605-0003,artotel suites aquila bandung|-6.8954|107.5893,2,hotel,same_type_duplicate_candidate,Review duplikat semantik; gabungkan hanya jika...,LODGE-00853,ARTOTEL Suites Aquila Bandung,artotel suites aquila bandung,hotel,-6.8954,107.5893,4.4,2172,620605,1.0,high,https://artotelwanderlust.com/hotel/artotel-su...,https://lh3.googleusercontent.com/gps-cs-s/APN...,2
5,DUP-20260605-0003,artotel suites aquila bandung|-6.8954|107.5893,2,hotel,same_type_duplicate_candidate,Review duplikat semantik; gabungkan hanya jika...,LODGE-01606,ARTOTEL Suites Aquila Bandung,artotel suites aquila bandung,hotel,-6.8954,107.5893,4.4,2170,,0.6,medium,https://www.google.com/aclk?sa=l&ai=DChsSEwjGy...,,1
6,DUP-20260605-0004,aubrey villa ciwidey chalet|-7.124|107.5054,3,villa,same_type_duplicate_candidate,Review duplikat semantik; gabungkan hanya jika...,LODGE-00525,Aubrey Villa Ciwidey - Chalet,aubrey villa ciwidey chalet,villa,-7.124,107.5054,,,254919,0.85,high,https://www.bluepillow.com/search/59439a957c00...,https://q-xx.bstatic.com/xdata/images/hotel/ma...,3
7,DUP-20260605-0004,aubrey villa ciwidey chalet|-7.124|107.5054,3,villa,same_type_duplicate_candidate,Review duplikat semantik; gabungkan hanya jika...,LODGE-00558,Aubrey Villa Ciwidey - Chalet,aubrey villa ciwidey chalet,villa,-7.124,107.5054,,,254734,0.8,high,https://www.freecancellations.com/search/F2giQ...,https://freecancellationscdn-hsbgcwfragasdpfe....,7
8,DUP-20260605-0004,aubrey villa ciwidey chalet|-7.124|107.5054,3,villa,same_type_duplicate_candidate,Review duplikat semantik; gabungkan hanya jika...,LODGE-00559,Aubrey Villa Ciwidey - Chalet,aubrey villa ciwidey chalet,villa,-7.124,107.5054,,,254919,0.8,high,https://www.bluepillow.com/search/59439a957c00...,https://q-xx.bstatic.com/xdata/images/hotel/ma...,3
9,DUP-20260605-0005,bantal guling trans|-6.9302|107.6395,3,room_level_listing,room_variant_group,Kelompok room/unit; cari parent utama atau tah...,LODGE-00874,Bantal Guling Trans - Standard Family Room,bantal guling trans,room_level_listing,-6.9302,107.6395,2.5,8,333261,1.0,high,https://www.bluepillow.com/search/654cb6b13011...,https://static.cupid.travel/hotels/248139690.jpg,1


### Keputusan Tahap B

Possible duplicate tidak digabung otomatis. File ini dipakai sebagai daftar review karena nama dan koordinat mirip belum selalu berarti properti yang sama.

## Tahap C - Audit Room-Level Listing

**Tujuan kecil:** memisahkan listing kamar/unit agar tidak ikut ranking sebagai properti utama.

In [14]:
room_level_path = CURATED_DIR / "PENGINAPAN_ROOM_LEVEL_LISTINGS_2026-06-05.csv"
room_level_df = pd.read_csv(room_level_path, dtype=str, keep_default_na=False)

display(
    pd.DataFrame(
        {
            "metric": [
                "Room-level listing rows",
                "Rows with duplicate group",
                "Rows name looks room/unit",
            ],
            "value": [
                len(room_level_df),
                int(room_level_df["possible_duplicate_group_id"].ne("").sum()),
                int(room_level_df["name_looks_room_or_unit"].eq("True").sum()),
            ],
        }
    )
)
display(room_level_df.head(40))

,metric,value
0,Room-level listing rows,54
1,Rows with duplicate group,23
2,Rows name looks room/unit,49


,child_penginapan_id,child_name,child_normalized_name,child_property_type,latitude,longitude,overall_rating,reviews,price_lowest,data_quality_score,data_quality_label,possible_duplicate_group_id,name_looks_room_or_unit,suggested_handling,parent_search_hint,link,primary_image_url
0,LODGE-00022,OYO 3394 Apm Residence Syariah - Deluxe Double...,oyo 3394 apm residence syariah,room_level_listing,-6.8705,107.0593,,,,0.6,medium,,True,keep_as_child_candidate_not_main_ranking,Cari parent dari nama dasar + koordinat sama/d...,https://www.bluepillow.com/search/67d167c3a21d...,https://static.cupid.travel/hotels/ex_c49c07d7...
1,LODGE-00065,Hotel O Cibeureum Residence - Deluxe Double Room,hotel o cibeureum residence,room_level_listing,-6.9106,107.5689,2.75,2,194180,0.95,high,,True,keep_as_child_candidate_not_main_ranking,Cari parent dari nama dasar + koordinat sama/d...,https://www.bluepillow.com/search/5df4f573e24d...,https://q-xx.bstatic.com/xdata/images/hotel/ma...
2,LODGE-00087,Budi House Food Station - Deluxe Twin Room,budi house food station,room_level_listing,-6.8908,107.559,1.85,3,302236,0.9,high,DUP-20260605-0006,True,keep_as_child_candidate_not_main_ranking,Cari parent dari nama dasar + koordinat sama/d...,https://www.bluepillow.com/search/594386f87c00...,https://q-xx.bstatic.com/xdata/images/hotel/ma...
3,LODGE-00088,Budi House Food Station - Standard Double Room,budi house food station,room_level_listing,-6.8908,107.559,1.85,3,356798,0.9,high,DUP-20260605-0006,True,keep_as_child_candidate_not_main_ranking,Cari parent dari nama dasar + koordinat sama/d...,https://www.bluepillow.com/search/594386f87c00...,https://static.cupid.travel/hotels/ex_d396a364...
4,LODGE-00120,Sans Stay Skyland Pasteur Bandung - Deluxe Dou...,sans stay skyland pasteur bandung,room_level_listing,-6.8829,107.5794,,,188740,0.85,high,DUP-20260605-0062,True,keep_as_child_candidate_not_main_ranking,Cari parent dari nama dasar + koordinat sama/d...,https://www.bluepillow.com/search/6949716c5c04...,https://q-xx.bstatic.com/xdata/images/hotel/ma...
5,LODGE-00121,Sans Stay Skyland Pasteur Bandung - Standard D...,sans stay skyland pasteur bandung,room_level_listing,-6.8829,107.5794,,,190531,0.85,high,DUP-20260605-0062,True,keep_as_child_candidate_not_main_ranking,Cari parent dari nama dasar + koordinat sama/d...,https://www.bluepillow.com/search/6949716c5c04...,https://q-xx.bstatic.com/xdata/images/hotel/ma...
6,LODGE-00150,Budi House Food Station - Superior Double Room,budi house food station,room_level_listing,-6.8908,107.559,1.85,6,283335,0.8,high,DUP-20260605-0006,True,keep_as_child_candidate_not_main_ranking,Cari parent dari nama dasar + koordinat sama/d...,,https://static.cupid.travel/hotels/ex_d396a364...
7,LODGE-00206,OYO 2088 Grha Blue Sky Syariah - Standard Doub...,oyo 2088 grha blue sky syariah,room_level_listing,-6.8843,107.5781,,,106094,0.75,high,,True,keep_as_child_candidate_not_main_ranking,Cari parent dari nama dasar + koordinat sama/d...,https://www.bluepillow.com/search/67d16617a21d...,https://static.cupid.travel/hotels/ex_fb24067c...
8,LODGE-00239,Little Blue House - Queen Room,little blue house,room_level_listing,-6.8452,107.5457,,,,0.7,medium,DUP-20260605-0036,True,keep_as_child_candidate_not_main_ranking,Cari parent dari nama dasar + koordinat sama/d...,https://www.bluepillow.com/search/6626441a4338...,https://q-xx.bstatic.com/xdata/images/hotel/ma...
9,LODGE-00257,ZEN Rooms Cibogo Pasteur,zen rooms cibogo pasteur,room_level_listing,-6.889233,107.579627,4.1,21,,0.68,medium,,False,keep_as_child_candidate_not_main_ranking,Cari parent dari nama dasar + koordinat sama/d...,https://www.zenrooms.com/id/hotel/8/zen-rooms-...,https://lh3.googleusercontent.com/gps-cs-s/APN...


### Keputusan Tahap C

Room-level listing tetap disimpan, tetapi disiapkan sebagai child candidate. Data ini tidak boleh dipakai sebagai daftar properti utama untuk ranking.

## Tahap D - Parent-Child Mapping Candidate

**Tujuan kecil:** mencari kandidat parent untuk room-level listing dengan aturan konservatif.

Catatan: confidence tinggi tetap belum final. Ini baru kandidat untuk direview.

In [15]:
mapping_path = CURATED_DIR / "PENGINAPAN_PARENT_CHILD_MAPPING_CANDIDATE_2026-06-05.csv"
mapping_df = pd.read_csv(mapping_path, dtype=str, keep_default_na=False)

display(
    pd.DataFrame(
        {
            "metric": [
                "Mapping candidate rows",
                "Child with parent candidate",
                "Child without parent candidate",
            ],
            "value": [
                len(mapping_df),
                mapping_df[mapping_df["parent_candidate_id"].ne("")]["child_penginapan_id"].nunique(),
                mapping_df[mapping_df["parent_candidate_id"].eq("")]["child_penginapan_id"].nunique(),
            ],
        }
    )
)
display(mapping_df["confidence_label"].value_counts().rename_axis("confidence_label").reset_index(name="rows"))
display(
    mapping_df[
        [
            "child_penginapan_id",
            "child_name",
            "parent_candidate_id",
            "parent_candidate_name",
            "confidence_score",
            "confidence_label",
            "match_basis",
            "decision_recommendation",
        ]
    ].head(60)
)

,metric,value
0,Mapping candidate rows,54
1,Child with parent candidate,13
2,Child without parent candidate,41


,confidence_label,rows
0,no_candidate,41
1,high,8
2,medium,5


,child_penginapan_id,child_name,parent_candidate_id,parent_candidate_name,confidence_score,confidence_label,match_basis,decision_recommendation
0,LODGE-00022,OYO 3394 Apm Residence Syariah - Deluxe Double...,,,0.0,no_candidate,no_safe_parent_candidate,manual_review_or_keep_child_without_parent
1,LODGE-00065,Hotel O Cibeureum Residence - Deluxe Double Room,,,0.0,no_candidate,no_safe_parent_candidate,manual_review_or_keep_child_without_parent
2,LODGE-00087,Budi House Food Station - Deluxe Twin Room,,,0.0,no_candidate,no_safe_parent_candidate,manual_review_or_keep_child_without_parent
3,LODGE-00088,Budi House Food Station - Standard Double Room,,,0.0,no_candidate,no_safe_parent_candidate,manual_review_or_keep_child_without_parent
4,LODGE-00120,Sans Stay Skyland Pasteur Bandung - Deluxe Dou...,LODGE-00048,Sans Hotel Skyland Pasteur Bandung,0.88,high,very_close_coordinate_and_strong_name_overlap,candidate_mapping_review_before_final
5,LODGE-00121,Sans Stay Skyland Pasteur Bandung - Standard D...,LODGE-00048,Sans Hotel Skyland Pasteur Bandung,0.88,high,very_close_coordinate_and_strong_name_overlap,candidate_mapping_review_before_final
6,LODGE-00150,Budi House Food Station - Superior Double Room,,,0.0,no_candidate,no_safe_parent_candidate,manual_review_or_keep_child_without_parent
7,LODGE-00206,OYO 2088 Grha Blue Sky Syariah - Standard Doub...,LODGE-00082,Grha Blue Sky,0.78,medium,close_coordinate_and_name_similarity,candidate_mapping_review_before_final
8,LODGE-00239,Little Blue House - Queen Room,,,0.0,no_candidate,no_safe_parent_candidate,manual_review_or_keep_child_without_parent
9,LODGE-00257,ZEN Rooms Cibogo Pasteur,,,0.0,no_candidate,no_safe_parent_candidate,manual_review_or_keep_child_without_parent


### Keputusan Tahap D

Relasi parent-child belum difinalkan. Baris dengan parent candidate dapat direview lebih dulu, sedangkan child tanpa parent tetap ditahan sebagai child candidate tanpa parent.

## Tahap E - Verifikasi Output Fase B

**Tujuan kecil:** memastikan semua file hasil entity resolution candidate sudah tersedia.

In [16]:
phase_b_outputs = {
    "possible_duplicate_review": CURATED_DIR / "PENGINAPAN_POSSIBLE_DUPLICATE_REVIEW_2026-06-05.csv",
    "room_level_listings": CURATED_DIR / "PENGINAPAN_ROOM_LEVEL_LISTINGS_2026-06-05.csv",
    "parent_child_mapping_candidate": CURATED_DIR / "PENGINAPAN_PARENT_CHILD_MAPPING_CANDIDATE_2026-06-05.csv",
    "summary": CURATED_DIR / "penginapan_parent_child_candidate_summary_2026-06-05.json",
}

artifact_rows = []
for name, path in phase_b_outputs.items():
    rows = None
    columns = None
    if path.exists() and path.suffix == ".csv":
        temp = pd.read_csv(path, dtype=str, keep_default_na=False)
        rows = len(temp)
        columns = len(temp.columns)
    artifact_rows.append({"artifact": name, "exists": path.exists(), "rows": rows, "columns": columns, "path": str(path)})

display(pd.DataFrame(artifact_rows))

,artifact,exists,rows,columns,path
0,possible_duplicate_review,True,247.0,20.0,D:\File\file\Fauzan Lubada\PIJAK\Penginapan_Wo...
1,room_level_listings,True,54.0,17.0,D:\File\file\Fauzan Lubada\PIJAK\Penginapan_Wo...
2,parent_child_mapping_candidate,True,54.0,20.0,D:\File\file\Fauzan Lubada\PIJAK\Penginapan_Wo...
3,summary,True,NaN,NaN,D:\File\file\Fauzan Lubada\PIJAK\Penginapan_Wo...


### Keputusan Tahap E

Fase B selesai sebagai candidate mapping. Langkah berikutnya adalah review relasi parent-child yang sudah ada kandidatnya, lalu baru membuat target scraping review dari properti utama.


## Catatan Review Manual - Parent-Child Candidate

Review manual untuk kandidat nomor **1-13** menyatakan bahwa pasangan tersebut merujuk ke properti/lokasi yang sama.

Perbedaan nama berasal dari format OTA: **nama properti + tipe kamar**. Jadi label seperti `Deluxe Double Room`, `Standard Double Room`, `Family Room`, `King Room`, `Twin Room`, atau variasi kamar lain dipahami sebagai unit kamar, bukan tempat berbeda.

Keputusan manual:

| No | Keputusan |
|---:|---|
| 1 | Accept - child milik Sans Hotel Skyland Pasteur Bandung |
| 2 | Accept - child milik Sans Hotel Skyland Pasteur Bandung |
| 3 | Accept - child milik Grha Blue Sky |
| 4 | Accept - child milik Vila Kopi Ciwidey |
| 5 | Accept - child milik Vila Kopi Ciwidey |
| 6 | Accept - child milik Vila Kopi Ciwidey |
| 7 | Accept - child milik Bantal Guling Guesthouse Trans |
| 8 | Accept - child milik Bantal Guling Guesthouse Trans |
| 9 | Accept - child milik Bantal Guling Guesthouse Trans |
| 10 | Accept - child milik RedDoorz Syariah near BTC Fashion Mall |
| 11 | Accept - child milik Bantal Guling Guesthouse Trans |
| 12 | Accept - child milik RedDoorz Plus near Asia Afrika 3 |
| 13 | Accept - child milik Penginapan Rio Anakku Syariah Banjaran Mitra RedDoorz |

Catatan: keputusan ini berasal dari review manual dan pengecekan Google, bukan dari auto-merge final. Relasi final tetap dibuat sebagai tahap terpisah agar jejak keputusan jelas.


## Fase C - Parent Master dan Target Review

Fase ini memakai 13 relasi yang sudah direview manual sebagai relasi final.

| Catatan |
|---|
| Child tidak dibuang. Child hanya dipisahkan dari daftar parent utama. |
| Parent master dipakai untuk ranking dan target scraping review. |

## Tahap A - Buat Relasi Final

| Catatan |
|---|
| 13 pasangan yang sudah accept dicatat sebagai relasi final. |

In [17]:
from pathlib import Path
import json
import subprocess
import sys

import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT.name != "PIJAK" and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent

WORKSPACE = PROJECT_ROOT / "Penginapan_Workspace"
CURATED_DIR = WORKSPACE / "02_Curated"
SCRIPT_DIR = WORKSPACE / "06_Scripts"

run_final = subprocess.run(
    [sys.executable, str(SCRIPT_DIR / "build_penginapan_parent_master_final.py")],
    check=True,
    capture_output=True,
    text=True,
)
print(run_final.stdout.strip())

relations_path = CURATED_DIR / "PENGINAPAN_PARENT_CHILD_RELATIONS_FINAL_2026-06-05.csv"
relations_df = pd.read_csv(relations_path, dtype=str, keep_default_na=False)

display(pd.DataFrame({"metric": ["Final relations"], "value": [len(relations_df)]}))
display(relations_df[["child_penginapan_id", "child_name", "parent_penginapan_id", "parent_name", "manual_decision"]])

relations_final_rows=13
parent_master_rows=1602
child_listings_rows=54
review_target_rows=1518
validation_gate=PASS_WITH_WARNINGS
accepted_child_in_parent_master=0
child_in_review_targets=0


,metric,value
0,Final relations,13


,child_penginapan_id,child_name,parent_penginapan_id,parent_name,manual_decision
0,LODGE-00120,Sans Stay Skyland Pasteur Bandung - Deluxe Dou...,LODGE-00048,Sans Hotel Skyland Pasteur Bandung,accept
1,LODGE-00121,Sans Stay Skyland Pasteur Bandung - Standard D...,LODGE-00048,Sans Hotel Skyland Pasteur Bandung,accept
2,LODGE-00206,OYO 2088 Grha Blue Sky Syariah - Standard Doub...,LODGE-00082,Grha Blue Sky,accept
3,LODGE-00546,Vila Kopi Ciwidey - Family Room with Garden View,LODGE-00627,Vila Kopi Ciwidey,accept
4,LODGE-00596,Vila Kopi Ciwidey - Family Room with Garden View,LODGE-00627,Vila Kopi Ciwidey,accept
5,LODGE-00597,Vila Kopi Ciwidey - King Room with Garden View,LODGE-00627,Vila Kopi Ciwidey,accept
6,LODGE-00873,Bantal Guling Trans - Standard Double Room wit...,LODGE-00871,Bantal Guling Guesthouse Trans,accept
7,LODGE-00874,Bantal Guling Trans - Standard Family Room,LODGE-00871,Bantal Guling Guesthouse Trans,accept
8,LODGE-00875,Bantal Guling Trans - Superior Family Room,LODGE-00871,Bantal Guling Guesthouse Trans,accept
9,LODGE-01113,Bantal Guling Trans - Family Triple Room,LODGE-00871,Bantal Guling Guesthouse Trans,accept


### Keputusan Tahap A

| Catatan |
|---|
| 13 relasi diterima sebagai final karena sudah direview manual. |
| Ini bukan auto-merge; sumber keputusannya tetap dicatat. |

## Tahap B - Pisahkan Parent dan Child

| Catatan |
|---|
| Parent untuk ranking. Child untuk detail kamar/unit. |

In [18]:
summary_path = CURATED_DIR / "penginapan_parent_master_final_summary_2026-06-05.json"
summary = json.loads(summary_path.read_text(encoding="utf-8"))

parent_master_path = CURATED_DIR / "PENGINAPAN_PARENT_MASTER_2026-06-05.csv"
child_final_path = CURATED_DIR / "PENGINAPAN_CHILD_LISTINGS_FINAL_2026-06-05.csv"

parent_master_df = pd.read_csv(parent_master_path, dtype=str, keep_default_na=False)
child_final_df = pd.read_csv(child_final_path, dtype=str, keep_default_na=False)

display(
    pd.DataFrame(
        {
            "metric": [
                "Parent master rows",
                "Child listing rows",
                "Accepted child",
                "Child tanpa parent final",
                "Parent dengan child final",
            ],
            "value": [
                summary["parent_master_rows"],
                summary["child_listings_rows"],
                summary["child_listing_status_counts"].get("accepted_parent", 0),
                summary["child_listing_status_counts"].get("child_without_final_parent", 0),
                summary["parent_with_child_count"],
            ],
        }
    )
)
display(child_final_df[["child_penginapan_id", "child_name", "parent_penginapan_id", "parent_name", "relation_status"]].head(30))

,metric,value
0,Parent master rows,1602
1,Child listing rows,54
2,Accepted child,13
3,Child tanpa parent final,41
4,Parent dengan child final,7


,child_penginapan_id,child_name,parent_penginapan_id,parent_name,relation_status
0,LODGE-00022,OYO 3394 Apm Residence Syariah - Deluxe Double...,,,child_without_final_parent
1,LODGE-00065,Hotel O Cibeureum Residence - Deluxe Double Room,,,child_without_final_parent
2,LODGE-00087,Budi House Food Station - Deluxe Twin Room,,,child_without_final_parent
3,LODGE-00088,Budi House Food Station - Standard Double Room,,,child_without_final_parent
4,LODGE-00120,Sans Stay Skyland Pasteur Bandung - Deluxe Dou...,LODGE-00048,Sans Hotel Skyland Pasteur Bandung,accepted_parent
5,LODGE-00121,Sans Stay Skyland Pasteur Bandung - Standard D...,LODGE-00048,Sans Hotel Skyland Pasteur Bandung,accepted_parent
6,LODGE-00150,Budi House Food Station - Superior Double Room,,,child_without_final_parent
7,LODGE-00206,OYO 2088 Grha Blue Sky Syariah - Standard Doub...,LODGE-00082,Grha Blue Sky,accepted_parent
8,LODGE-00239,Little Blue House - Queen Room,,,child_without_final_parent
9,LODGE-00257,ZEN Rooms Cibogo Pasteur,,,child_without_final_parent


### Keputusan Tahap B

| Catatan |
|---|
| Parent master sudah terpisah dari child listing. |
| 41 child tanpa parent final tetap ditahan sebagai child, bukan dipromosikan ke parent. |

## Tahap C - Validasi Pemisahan

| Catatan |
|---|
| Yang dicek: accepted child tidak boleh muncul di parent master dan target review. |

In [19]:
validation_path = CURATED_DIR / "penginapan_parent_master_final_validation_2026-06-05.json"
validation = json.loads(validation_path.read_text(encoding="utf-8"))

display(
    pd.DataFrame(
        {
            "metric": [
                "Validation gate",
                "Accepted child in parent master",
                "Any child in parent master",
                "Child in review targets",
                "Child without final parent",
            ],
            "value": [
                validation["gate"],
                validation["accepted_child_in_parent_master"],
                validation["any_child_in_parent_master"],
                validation["child_in_review_targets"],
                validation["child_without_final_parent"],
            ],
        }
    )
)
display(pd.DataFrame(validation["warnings"]))

,metric,value
0,Validation gate,PASS_WITH_WARNINGS
1,Accepted child in parent master,0
2,Any child in parent master,0
3,Child in review targets,0
4,Child without final parent,41


,code,count,message
0,CHILD_WITHOUT_FINAL_PARENT,41,"Child tetap ditahan dari parent master, tetapi..."


### Keputusan Tahap C

| Catatan |
|---|
| Validasi lolos dengan warning. Warning-nya wajar karena 41 child belum punya parent final. |
| Tidak ada child yang masuk parent master atau target review. |

## Tahap D - Target Scraping Review dari Parent Master

| Catatan |
|---|
| Target review dibuat dari parent master, bukan dari child listing. |
| Nama yang masih terlihat seperti unit diberi flag agar dicek saat batch test. |

In [20]:
review_targets_path = CURATED_DIR / "PENGINAPAN_REVIEW_SCRAPE_TARGETS_PARENT_MASTER_2026-06-05.csv"
review_targets_df = pd.read_csv(review_targets_path, dtype=str, keep_default_na=False)

display(
    pd.DataFrame(
        {
            "metric": [
                "Review target rows",
                "Target flagged detail-name",
                "Missing rating/review",
                "Low review confidence",
                "Medium review confidence",
                "High review confidence",
            ],
            "value": [
                len(review_targets_df),
                summary["review_target_detail_listing_flag_count"],
                summary["review_target_priority_counts"].get("missing_rating_or_review", 0),
                summary["review_target_priority_counts"].get("low_review_confidence", 0),
                summary["review_target_priority_counts"].get("medium_review_confidence", 0),
                summary["review_target_priority_counts"].get("high_review_confidence", 0),
            ],
        }
    )
)
display(review_targets_df[["review_target_id", "penginapan_id", "name", "property_type", "review_scrape_priority", "name_looks_detail_listing", "target_review_note"]].head(40))

,metric,value
0,Review target rows,1518
1,Target flagged detail-name,387
2,Missing rating/review,599
3,Low review confidence,378
4,Medium review confidence,200
5,High review confidence,341


,review_target_id,penginapan_id,name,property_type,review_scrape_priority,name_looks_detail_listing,target_review_note
0,PHREV-20260605-0001,LODGE-00214,Airy Eco Budi Cimahi,hotel,high_review_confidence,False,parent_target
1,PHREV-20260605-0002,LODGE-01021,Rizh Hotel,hotel,high_review_confidence,False,parent_target
2,PHREV-20260605-0003,LODGE-00881,Bumi Makhraja,hotel,high_review_confidence,False,parent_target
3,PHREV-20260605-0004,LODGE-00083,Stay at Rozelle,hotel,high_review_confidence,False,parent_target
4,PHREV-20260605-0005,LODGE-00776,LuxCamp Riverside Pangalengan by Horison,hotel,high_review_confidence,False,parent_target
5,PHREV-20260605-0006,LODGE-00890,Collection O Kebon Jeruk Near Paskal 23 Former...,hotel,high_review_confidence,False,parent_target
6,PHREV-20260605-0007,LODGE-00048,Sans Hotel Skyland Pasteur Bandung,hotel,high_review_confidence,False,parent_target
7,PHREV-20260605-0008,LODGE-00992,RedDoorz Plus near Kampus UPI Setiabudhi,hotel,high_review_confidence,False,parent_target
8,PHREV-20260605-0009,LODGE-01579,RedDoorz @ Talaga Bodas,hotel,high_review_confidence,False,parent_target
9,PHREV-20260605-0010,LODGE-01183,Sartika Hotel,hotel,high_review_confidence,False,parent_target


### Keputusan Tahap D

| Catatan |
|---|
| Target review sudah bisa dipakai untuk batch test kecil dulu. |
| Target dengan flag detail-name jangan langsung dipercaya; cek hasil match Google Maps saat test. |

## Fase D - Split Target Review Sebelum Scraping

Bagian ini memakai **Format Notebook Audit Bertahap**. Tujuannya sederhana: target scraping review dipisahkan dulu agar kamar/unit tidak ikut discrape sebagai parent.

### Proses A - Audit Kelompok Flagged

Tujuan kecil: cek ulang berapa target review yang namanya masih terlihat seperti kamar, unit, atau detail listing.

In [21]:
from pathlib import Path
import pandas as pd

CURATED_DIR = Path("../02_Curated")
if not CURATED_DIR.exists():
    CURATED_DIR = Path("Penginapan_Workspace/02_Curated")

targets = pd.read_csv(CURATED_DIR / "PENGINAPAN_REVIEW_SCRAPE_TARGETS_PARENT_MASTER_2026-06-05.csv")
flagged = pd.read_csv(CURATED_DIR / "PENGINAPAN_REVIEW_TARGETS_FLAGGED_DETAIL_NAME_2026-06-05.csv")

summary_a = pd.DataFrame(
    [
        {"metric": "total_review_targets", "jumlah": len(targets)},
        {"metric": "flagged_detail_name_rows", "jumlah": len(flagged)},
    ]
)
flag_group_counts = (
    flagged["flag_group"]
    .value_counts(dropna=False)
    .rename_axis("flag_group")
    .reset_index(name="jumlah")
)

print("Ringkasan target review")
print(summary_a.to_string(index=False))
print("\nJumlah per flag_group")
print(flag_group_counts.to_string(index=False))

Ringkasan target review
                  metric  jumlah
    total_review_targets    1518
flagged_detail_name_rows     387

Jumlah per flag_group
                  flag_group  jumlah
hold_unit_or_bedroom_listing     265
  review_villa_or_house_unit      66
             hold_room_label      35
    review_other_detail_name      21

### Keputusan Proses A

Output menunjukkan ada 1.518 target review dan 387 nama yang terkena flag detail. Dua kelompok sudah jelas ditahan sebagai child: `hold_room_label` 35 data dan `hold_unit_or_bedroom_listing` 265 data. Dua kelompok lain masih perlu dicek ringan: 66 data villa/house dan 21 data detail lain.

### Proses B - Split Target Review Awal

Tujuan kecil: memisahkan target menjadi `ready`, `held_child`, dan `needs_review` berdasarkan kelompok flagged.

In [22]:
import json

summary_path = CURATED_DIR / "penginapan_review_targets_split_summary_2026-06-05.json"
with summary_path.open(encoding="utf-8") as handle:
    split_summary = json.load(handle)

summary_b = pd.DataFrame(
    [
        {"kelompok": "ready", "jumlah": split_summary["ready_rows"]},
        {"kelompok": "held_child", "jumlah": split_summary["held_child_rows"]},
        {"kelompok": "needs_review", "jumlah": split_summary["needs_review_rows"]},
    ]
)

output_files = pd.DataFrame(
    [
        {
            "file": Path(path).name,
            "rows": pd.read_csv(path).shape[0],
        }
        for path in split_summary["outputs"].values()
    ]
)

print("Hasil split target review")
print(summary_b.to_string(index=False))
print("\nFile output")
print(output_files.to_string(index=False))

Hasil split target review
    kelompok  jumlah
       ready    1131
  held_child     300
needs_review      87

File output
                                                       file  rows
             PENGINAPAN_REVIEW_TARGETS_READY_2026-06-05.csv  1131
        PENGINAPAN_REVIEW_TARGETS_HELD_CHILD_2026-06-05.csv   300
      PENGINAPAN_REVIEW_TARGETS_NEEDS_REVIEW_2026-06-05.csv    87
PENGINAPAN_REVIEW_TARGETS_NEEDS_REVIEW_AUDIT_2026-06-05.csv    87

### Keputusan Proses B

Split awal diterima. Ada 1.131 target yang aman untuk scraping review, 300 data ditahan sebagai child/detail, dan 87 data tidak dipaksa masuk parent karena masih abu-abu.

### Proses C - Audit 87 Data Abu-Abu

Tujuan kecil: melihat apakah 87 data abu-abu punya kandidat parent dari dataset saat ini, memakai nama dan kandidat koordinat yang tersedia.

In [23]:
needs_review_audit = pd.read_csv(
    CURATED_DIR / "PENGINAPAN_REVIEW_TARGETS_NEEDS_REVIEW_AUDIT_2026-06-05.csv"
)

audit_counts = (
    needs_review_audit["audit_recommendation"]
    .value_counts(dropna=False)
    .rename_axis("audit_recommendation")
    .reset_index(name="jumlah")
)
sample_cols = [
    "review_target_id",
    "name",
    "property_type",
    "flag_group",
    "audit_recommendation",
    "parent_candidate_name",
    "parent_candidate_score",
]

print("Ringkasan audit 87 data")
print(audit_counts.to_string(index=False))
print("\nContoh kandidat dari audit")
print(needs_review_audit[sample_cols].head(10).to_string(index=False))

Ringkasan audit 87 data
           audit_recommendation  jumlah
candidate_standalone_or_unclear      58
      candidate_child_to_parent      20
             needs_manual_check       9

Contoh kandidat dari audit
   review_target_id                                                                                  name   property_type                 flag_group      audit_recommendation                                                   parent_candidate_name  parent_candidate_score
PHREV-20260605-1248 Redliving Apartement gateway Pasteur-TN Hospitality Tower Topaz C Bs 01 - Deluxe Room       apartment   review_other_detail_name candidate_child_to_parent Redliving Apartement gateway Pasteur-TN Hospitality Tower Topaz C Bs 01                     1.0
PHREV-20260605-0598                                  Farmstay Manangel - Double Room with Shared Bathroom vacation_rental   review_other_detail_name candidate_child_to_parent                                                       Farmstay Manangel

### Keputusan Proses C

Dari 87 data abu-abu, 20 terlihat punya kandidat parent yang cukup kuat, 9 masih perlu cek manual, dan 58 belum punya kandidat parent yang kuat. Untuk saat ini 87 data ini tetap tidak masuk target scraping massal sampai diputuskan lebih lanjut.

## Fase E - Keputusan Lanjutan untuk 87 Data

Bagian ini mencatat keputusan setelah 87 data abu-abu dipisahkan. Tujuannya agar 58 data tidak hilang, tapi juga tidak menahan proses scraping utama.

### Proses A - Baca Hasil Audit 87 Data

Tujuan kecil: lihat ulang jumlah data berdasarkan rekomendasi audit.

In [24]:
from pathlib import Path
import json
import pandas as pd

CURATED_DIR = Path("../02_Curated")
if not CURATED_DIR.exists():
    CURATED_DIR = Path("Penginapan_Workspace/02_Curated")

audit_87 = pd.read_csv(CURATED_DIR / "PENGINAPAN_REVIEW_TARGETS_NEEDS_REVIEW_AUDIT_2026-06-05.csv")
audit_counts = (
    audit_87["audit_recommendation"]
    .value_counts(dropna=False)
    .rename_axis("audit_recommendation")
    .reset_index(name="jumlah")
)

print("Jumlah keputusan awal dari 87 data")
print(audit_counts.to_string(index=False))

Jumlah keputusan awal dari 87 data
           audit_recommendation  jumlah
candidate_standalone_or_unclear      58
      candidate_child_to_parent      20
             needs_manual_check       9

### Keputusan Proses A

Output membagi 87 data menjadi tiga arah: kandidat child-parent, cek manual, dan held low priority. Data ini tidak masuk scraping massal dulu.

### Proses B - Pisahkan Keputusan 20 / 9 / 58

Tujuan kecil: buat file kerja terpisah agar tindak lanjutnya tidak tercampur.

In [25]:
summary_path = CURATED_DIR / "penginapan_needs_review_decision_split_summary_2026-06-05.json"
with summary_path.open(encoding="utf-8") as handle:
    decision_summary = json.load(handle)

split_table = pd.DataFrame(
    [
        {"kelompok": "child_parent_candidate", "jumlah": decision_summary["child_parent_candidate_rows"]},
        {"kelompok": "manual_check", "jumlah": decision_summary["manual_check_rows"]},
        {"kelompok": "held_low_priority", "jumlah": decision_summary["held_low_priority_rows"]},
    ]
)

output_table = pd.DataFrame(
    [
        {"file": Path(path).name, "rows": pd.read_csv(path).shape[0]}
        for path in decision_summary["outputs"].values()
    ]
)

print("Hasil split 87 data")
print(split_table.to_string(index=False))
print("\nFile output")
print(output_table.to_string(index=False))

Hasil split 87 data
              kelompok  jumlah
child_parent_candidate      20
          manual_check       9
     held_low_priority      58

File output
                                                            file  rows
PENGINAPAN_NEEDS_REVIEW_20_CHILD_PARENT_CANDIDATE_2026-06-05.csv    20
           PENGINAPAN_NEEDS_REVIEW_9_MANUAL_CHECK_2026-06-05.csv     9
     PENGINAPAN_NEEDS_REVIEW_58_HELD_LOW_PRIORITY_2026-06-05.csv    58

### Keputusan Proses B

20 data masuk kandidat relasi child-parent. 9 data menunggu cek manual. 58 data ditahan sebagai low priority, bukan dihapus.

### Proses C - Validasi Split

Tujuan kecil: pastikan total tetap 87 dan tidak ada `review_target_id` yang masuk dua file.

In [26]:
files = {
    "child_parent_candidate": CURATED_DIR / "PENGINAPAN_NEEDS_REVIEW_20_CHILD_PARENT_CANDIDATE_2026-06-05.csv",
    "manual_check": CURATED_DIR / "PENGINAPAN_NEEDS_REVIEW_9_MANUAL_CHECK_2026-06-05.csv",
    "held_low_priority": CURATED_DIR / "PENGINAPAN_NEEDS_REVIEW_58_HELD_LOW_PRIORITY_2026-06-05.csv",
}

ids = {}
for key, path in files.items():
    df = pd.read_csv(path, dtype=str, keep_default_na=False)
    ids[key] = set(df["review_target_id"])
    print(f"{key}: rows={len(df)}, unique_ids={df['review_target_id'].nunique()}")

overlap = 0
keys = list(ids)
for index, left in enumerate(keys):
    for right in keys[index + 1:]:
        overlap += len(ids[left] & ids[right])

print(f"total_rows={sum(len(value) for value in ids.values())}")
print(f"overlap_review_target_id={overlap}")

child_parent_candidate: rows=20, unique_ids=20
manual_check: rows=9, unique_ids=9
held_low_priority: rows=58, unique_ids=58
total_rows=87
overlap_review_target_id=0

### Keputusan Proses C

Validasi diterima. Total kembali ke 87 dan tidak ada overlap, jadi keputusan 20 / 9 / 58 bisa dilacak dengan aman.

## Fase F - Keputusan Final 29 Data

Hasil review 29 data sudah ditutup. Bagian ini hanya mencatat keputusan finalnya agar alur parent/child tetap jelas.

### Proses A - Catat Keputusan Final

Tujuan kecil: baca hasil final 29 data dan cek jumlah tiap keputusan.

In [27]:
from pathlib import Path
import json
import pandas as pd

CURATED_DIR = Path("../02_Curated")
if not CURATED_DIR.exists():
    CURATED_DIR = Path("Penginapan_Workspace/02_Curated")

summary_path = CURATED_DIR / "penginapan_review_29_final_decision_summary_2026-06-05.json"
with summary_path.open(encoding="utf-8") as handle:
    final_29_summary = json.load(handle)

final_29 = pd.read_csv(CURATED_DIR / "PENGINAPAN_REVIEW_29_FINAL_DECISION_2026-06-05.csv")

summary_table = pd.DataFrame(
    [
        {"kelompok": "child_parent_relation", "jumlah": final_29_summary["child_parent_rows"]},
        {"kelompok": "parent_ready", "jumlah": final_29_summary["parent_ready_rows"]},
        {"kelompok": "held_drop", "jumlah": final_29_summary["held_drop_rows"]},
        {"kelompok": "needs_more_check", "jumlah": final_29_summary["needs_more_check_rows"]},
    ]
)

sample_cols = ["review_target_id", "name", "parent_candidate_name", "final_decision", "final_group"]
print("Ringkasan keputusan final 29 data")
print(summary_table.to_string(index=False))
print("\nContoh data final")
print(final_29[sample_cols].head(8).to_string(index=False))
print(f"\nfile_final={Path(final_29_summary['final_decision_path']).name}")

Ringkasan keputusan final 29 data
             kelompok  jumlah
child_parent_relation      19
         parent_ready       9
            held_drop       1
     needs_more_check       0

Contoh data final
   review_target_id                                                                                  name                                                   parent_candidate_name  final_decision           final_group
PHREV-20260605-1248 Redliving Apartement gateway Pasteur-TN Hospitality Tower Topaz C Bs 01 - Deluxe Room Redliving Apartement gateway Pasteur-TN Hospitality Tower Topaz C Bs 01 accept_as_child child_parent_relation
PHREV-20260605-1364                                                 The Edge By Orin - Deluxe Double Room                                                        The Edge By Orin accept_as_child child_parent_relation
PHREV-20260605-1365                                               The Edge By Orin - Standard Double Room                                            

### Keputusan Proses A

Keputusan final diterima. 19 data menjadi child-parent, 9 data tetap parent-ready, dan 1 data dikeluarkan dari jalur utama. Tidak ada data yang masih `needs_more_check`.

## Fase G - Pre-Scraping Target Review Final

Bagian ini menyiapkan target scraping review final. Target massal belum dijalankan; kita buat batch test kecil dulu.

### Proses A - Apply Keputusan Final

Tujuan kecil: terapkan keputusan final 29 data ke target review.

In [28]:
from pathlib import Path
import json
import pandas as pd

CURATED_DIR = Path("../02_Curated")
if not CURATED_DIR.exists():
    CURATED_DIR = Path("Penginapan_Workspace/02_Curated")

summary_path = CURATED_DIR / "penginapan_review_targets_final_summary_2026-06-05.json"
with summary_path.open(encoding="utf-8") as handle:
    final_target_summary = json.load(handle)

apply_table = pd.DataFrame(
    [
        {"metric": "initial_ready", "jumlah": final_target_summary["input_rows"]["initial_ready"]},
        {"metric": "final_29_parent_ready_added", "jumlah": final_target_summary["decision_rows"]["final_29_parent_ready"]},
        {"metric": "final_ready", "jumlah": final_target_summary["final_ready_rows"]},
        {"metric": "final_excluded", "jumlah": final_target_summary["final_excluded_rows"]},
    ]
)

excluded_table = (
    pd.DataFrame(
        final_target_summary["excluded_group_counts"].items(),
        columns=["excluded_group", "jumlah"],
    )
    .sort_values("excluded_group")
)

print("Ringkasan apply keputusan final")
print(apply_table.to_string(index=False))
print("\nExcluded group")
print(excluded_table.to_string(index=False))

Ringkasan apply keputusan final
                     metric  jumlah
              initial_ready    1131
final_29_parent_ready_added       9
                final_ready    1140
             final_excluded     378

Excluded group
       excluded_group  jumlah
final_29_child_parent      19
   final_29_held_drop       1
    held_child_detail     300
    held_low_priority      58

### Keputusan Proses A

Target ready final menjadi 1.140 data. Data yang ditahan berjumlah 378, terdiri dari child/detail, low priority, child-parent final, dan satu held/drop.

### Proses B - Validasi Ready dan Excluded

Tujuan kecil: pastikan target ready final tidak bercampur dengan data excluded.

In [29]:
final_ready = pd.read_csv(CURATED_DIR / "PENGINAPAN_REVIEW_TARGETS_FINAL_READY_2026-06-05.csv", dtype=str, keep_default_na=False)
final_excluded = pd.read_csv(CURATED_DIR / "PENGINAPAN_REVIEW_TARGETS_FINAL_EXCLUDED_2026-06-05.csv", dtype=str, keep_default_na=False)

ready_ids = set(final_ready["review_target_id"])
excluded_ids = set(final_excluded["review_target_id"])
overlap = len(ready_ids & excluded_ids)
duplicate_ready = final_ready["review_target_id"].duplicated().sum()
duplicate_excluded = final_excluded["review_target_id"].duplicated().sum()

validation_table = pd.DataFrame(
    [
        {"cek": "ready_rows", "hasil": len(final_ready)},
        {"cek": "excluded_rows", "hasil": len(final_excluded)},
        {"cek": "ready_excluded_overlap", "hasil": overlap},
        {"cek": "duplicate_ready_id", "hasil": int(duplicate_ready)},
        {"cek": "duplicate_excluded_id", "hasil": int(duplicate_excluded)},
    ]
)

print("Validasi target final")
print(validation_table.to_string(index=False))

Validasi target final
                   cek  hasil
            ready_rows   1140
         excluded_rows    378
ready_excluded_overlap      0
    duplicate_ready_id      0
 duplicate_excluded_id      0

### Keputusan Proses B

Validasi diterima. Tidak ada overlap antara ready dan excluded, dan tidak ada ID ganda di file target final.

### Proses C - Buat Batch Test 30 Target

Tujuan kecil: ambil batch kecil untuk cek hasil scraping sebelum jalan massal.

In [30]:
test_batch = pd.read_csv(CURATED_DIR / "PENGINAPAN_REVIEW_TARGETS_TEST_BATCH_30_2026-06-05.csv", dtype=str, keep_default_na=False)
sample_cols = [
    "review_target_id",
    "name",
    "property_type",
    "review_scrape_priority",
    "google_maps_search_query",
]
sample_cols = [col for col in sample_cols if col in test_batch.columns]

print(f"test_batch_rows={len(test_batch)}")
print("\nContoh target batch test")
print(test_batch[sample_cols].head(10).to_string(index=False))
print("\nfile_batch=PENGINAPAN_REVIEW_TARGETS_TEST_BATCH_30_2026-06-05.csv")

test_batch_rows=30

Contoh target batch test
   review_target_id                                                               name property_type review_scrape_priority                                                            google_maps_search_query
PHREV-20260605-0126                                            Apartemen Emerald Tower     apartment high_review_confidence                                            Apartemen Emerald Tower -6.9325,107.6643
PHREV-20260605-0076 Collection O Riau Near Trans Studio Bandung Formerly Asteria Hotel     apartment high_review_confidence Collection O Riau Near Trans Studio Bandung Formerly Asteria Hotel -6.9281,107.6214
PHREV-20260605-0036                           Gateway Apartment Cicadas by DB Property     apartment high_review_confidence                           Gateway Apartment Cicadas by DB Property -6.9067,107.6446
PHREV-20260605-0161                                   High Livin Apartment Asia Afrika     apartment high_review_confidence

### Keputusan Proses C

Batch test 30 target sudah dibuat. Scraping sebaiknya dimulai dari file ini dulu, bukan langsung dari semua target ready.

## Fase H - Sentiment Penginapan dan Ranking Jarak

Fase ini memakai model sentiment penginapan untuk membaca review hotel, lalu membuat baseline ranking dengan bobot jarak lebih besar.

### H1 - Cek Input

Tujuan: pastikan parent master, review normalized, dan model sentiment sudah tersedia.

In [1]:
from pathlib import Path
import json
import pandas as pd

ROOT = Path.cwd()
WORKSPACE = ROOT / "Penginapan_Workspace"
CURATED_DIR = WORKSPACE / "02_Curated"
MODEL_PATH = WORKSPACE / "07_Models" / "model_sentimen_muterbandung.pkl"

input_files = [
    CURATED_DIR / "PENGINAPAN_PARENT_MASTER_2026-06-05.csv",
    CURATED_DIR / "PENGINAPAN_COMPASS_REVIEW_TEST_BATCH_30_RAW_NORMALIZED_2026-06-06.csv",
    CURATED_DIR / "PENGINAPAN_PARENT_CHILD_RELATIONS_FINAL_2026-06-05.csv",
    MODEL_PATH,
]

rows = []
for path in input_files:
    rows.append({
        "file": str(path.relative_to(ROOT)),
        "exists": path.exists(),
        "size_mb": round(path.stat().st_size / 1024 / 1024, 2) if path.exists() else None,
    })
pd.DataFrame(rows)

,file,exists,size_mb
0,Penginapan_Workspace\02_Curated\PENGINAPAN_PAR...,True,1.65
1,Penginapan_Workspace\02_Curated\PENGINAPAN_COM...,True,1.94
2,Penginapan_Workspace\02_Curated\PENGINAPAN_PAR...,True,0.00
3,Penginapan_Workspace\07_Models\model_sentimen_...,True,2.36


Keputusan: input cukup untuk batch awal. Review yang dipakai masih batch test, jadi hasil sentiment belum mewakili semua penginapan.

### H2 - Jalankan Inference dan Agregasi

Tujuan: prediksi sentiment per review, lalu agregasi ke parent hotel.

In [2]:
import subprocess
import sys

script_path = WORKSPACE / "06_Scripts" / "build_penginapan_sentiment_baseline.py"
result = subprocess.run(
    [sys.executable, str(script_path)],
    cwd=ROOT,
    capture_output=True,
    text=True,
    check=True,
)
summary = json.loads(result.stdout)
print(json.dumps({
    "review_rows_input": summary["review_rows_input"],
    "review_rows_with_text_inference": summary["review_rows_with_text_inference"],
    "parent_hotels_with_sentiment": summary["parent_hotels_with_sentiment"],
    "parent_master_rows": summary["parent_master_rows"],
    "parent_master_with_sentiment_rows": summary["parent_master_with_sentiment_rows"],
    "distance_weight": summary["distance_weighted_baseline"]["weights"]["distance"],
    "sentiment_weight": summary["distance_weighted_baseline"]["weights"]["sentiment"],
}, indent=2, ensure_ascii=False))

{
  "review_rows_input": 971,
  "review_rows_with_text_inference": 626,
  "parent_hotels_with_sentiment": 28,
  "parent_master_rows": 1602,
  "parent_master_with_sentiment_rows": 28,
  "distance_weight": 0.45,
  "sentiment_weight": 0.16
}


Keputusan: sentiment berhasil masuk ke 28 parent hotel dari batch test. Ranking hotel dibuat distance-heavy karena user biasanya mencari penginapan dekat tujuan.

### H3 - Audit Agregasi Sentiment

Tujuan: lihat hasil sentiment per hotel yang sudah punya review teks.

In [3]:
aggregated_path = CURATED_DIR / "PENGINAPAN_SENTIMENT_AGGREGATED_2026-06-10.csv"
aggregated = pd.read_csv(aggregated_path)

display(
    aggregated[
        [
            "parent_penginapan_id",
            "hotel_review_count_analyzed",
            "hotel_sentiment_score",
            "hotel_adjusted_sentiment_score",
            "hotel_adjusted_sentiment_label",
            "hotel_review_confidence_label",
            "positive_review_count",
            "neutral_review_count",
            "negative_review_count",
        ]
    ]
    .sort_values(["hotel_review_count_analyzed", "hotel_adjusted_sentiment_score"], ascending=[False, False])
    .head(15)
)

aggregated["hotel_adjusted_sentiment_label"].value_counts().rename_axis("label").reset_index(name="count")

,parent_penginapan_id,hotel_review_count_analyzed,hotel_sentiment_score,hotel_adjusted_sentiment_score,hotel_adjusted_sentiment_label,hotel_review_confidence_label,positive_review_count,neutral_review_count,negative_review_count
12,LODGE-01018,94,0.224664,0.329418,Positif,high_review_confidence,58,1,35
18,LODGE-01295,44,0.848534,0.677162,Positif,high_review_confidence,44,0,0
8,LODGE-00861,26,-0.349076,0.226865,Positif,high_review_confidence,8,0,18
21,LODGE-01350,25,0.946413,0.666374,Positif,high_review_confidence,25,0,0
7,LODGE-00819,24,0.917648,0.653261,Positif,high_review_confidence,24,0,0
27,LODGE-01616,24,0.917232,0.653126,Positif,high_review_confidence,24,0,0
6,LODGE-00438,21,0.923501,0.643821,Positif,high_review_confidence,21,0,0
10,LODGE-00871,21,0.567859,0.538631,Positif,high_review_confidence,17,0,4
20,LODGE-01341,20,0.886018,0.629116,Positif,high_review_confidence,20,0,0
22,LODGE-01356,20,0.839236,0.615749,Positif,high_review_confidence,20,0,0


,label,count
0,Positif,28


Keputusan: hasil ini layak sebagai sinyal awal. Hotel tanpa review batch tetap diberi status sentiment belum tersedia, bukan dipaksa netral.

### H4 - Dataset Parent Dengan Sentiment

Tujuan: cek dataset parent master setelah kolom sentiment digabung.

In [4]:
parent_sentiment_path = CURATED_DIR / "PENGINAPAN_PARENT_MASTER_WITH_SENTIMENT_2026-06-10.csv"
parent_with_sentiment = pd.read_csv(parent_sentiment_path)

audit_parent = {
    "rows": len(parent_with_sentiment),
    "with_sentiment": int(parent_with_sentiment["hotel_sentiment_available"].sum()),
    "without_sentiment": int((~parent_with_sentiment["hotel_sentiment_available"].astype(bool)).sum()),
    "columns": len(parent_with_sentiment.columns),
}
print(json.dumps(audit_parent, indent=2, ensure_ascii=False))

parent_with_sentiment[
    [
        "penginapan_id",
        "name",
        "property_type",
        "overall_rating",
        "reviews",
        "price_lowest",
        "hotel_sentiment_available",
        "hotel_adjusted_sentiment_score",
        "hotel_adjusted_sentiment_label",
        "hotel_review_confidence_label",
    ]
].query("hotel_sentiment_available == True").head(10)

{
  "rows": 1602,
  "with_sentiment": 28,
  "without_sentiment": 1574,
  "columns": 67
}


,penginapan_id,name,property_type,overall_rating,reviews,price_lowest,hotel_sentiment_available,hotel_adjusted_sentiment_score,hotel_adjusted_sentiment_label,hotel_review_confidence_label
67,LODGE-00070,OYO 3463 Cimahi Guest House,guest_house,3.7,323.0,130628.0,True,0.448618,Positif,high_review_confidence
77,LODGE-00080,Cottonwood Bed & Breakfast House Bandung,guest_house,4.6,853.0,395152.0,True,0.569726,Positif,high_review_confidence
78,LODGE-00081,LN9 Bandung Guest House,guest_house,4.3,585.0,154062.0,True,0.440300,Positif,high_review_confidence
93,LODGE-00098,Saloka Guest House dan Residence,guest_house,4.3,407.0,187032.0,True,0.492837,Positif,medium_review_confidence
109,LODGE-00114,Micasa Guest House Bandung,guest_house,3.3,302.0,98135.0,True,0.318825,Positif,high_review_confidence
191,LODGE-00199,Cluster Harris Home Stay & Kos,guest_house,4.3,349.0,NaN,True,0.563380,Positif,high_review_confidence
427,LODGE-00438,Hen's Guest House,guest_house,4.6,245.0,NaN,True,0.643821,Positif,high_review_confidence
790,LODGE-00819,Jayagiri Guesthouse,guest_house,4.5,901.0,344423.0,True,0.653261,Positif,high_review_confidence
830,LODGE-00861,Apartemen Emerald Tower,apartment,3.8,559.0,126655.0,True,0.226865,Positif,high_review_confidence
839,LODGE-00870,Bantal Guling Guest House Alun-alun Bandung,guest_house,4.1,821.0,122364.0,True,0.581995,Positif,high_review_confidence


Keputusan: file ini menjadi kandidat runtime hotel berikutnya. Coverage sentiment masih kecil karena input review baru batch test.

### H5 - Baseline Ranking Dengan Jarak Lebih Besar

Tujuan: audit contoh ranking hotel saat jarak menjadi bobot terbesar.

In [5]:
sample_path = CURATED_DIR / "PENGINAPAN_DISTANCE_WEIGHTED_BASELINE_SAMPLE_2026-06-10.csv"
sample = pd.read_csv(sample_path)

display(
    sample[
        [
            "penginapan_id",
            "name",
            "property_type",
            "distance_km_reference",
            "hotel_recommendation_score",
            "hotel_distance_score",
            "hotel_rating_score",
            "hotel_sentiment_ranking_score",
            "hotel_price_score",
            "hotel_review_count_analyzed",
            "hotel_adjusted_sentiment_label",
        ]
    ].head(20)
)

sample[["hotel_recommendation_score", "distance_km_reference"]].describe()

,penginapan_id,name,property_type,distance_km_reference,hotel_recommendation_score,hotel_distance_score,hotel_rating_score,hotel_sentiment_ranking_score,hotel_price_score,hotel_review_count_analyzed,hotel_adjusted_sentiment_label
0,LODGE-01295,Pension Guesthouse,guest_house,1.042922,86.330227,0.827414,0.92,0.838581,0.868809,44,Positif
1,LODGE-01350,De Halimun Guest House,guest_house,1.021684,84.969070,0.830333,0.88,0.833187,0.891899,25,Positif
2,LODGE-01257,High Livin Apartment Asia Afrika,apartment,0.728235,83.072197,0.872869,0.76,0.692203,0.984769,16,Positif
3,LODGE-00961,Mogens Guest House,guest_house,1.539621,82.666079,0.764570,0.92,0.789510,0.919584,15,Positif
4,LODGE-00958,Meize City Center Hotel Bandung,hotel,0.332759,81.031592,0.937601,0.84,0.500000,0.968839,0,Belum tersedia
5,LODGE-00936,Hotel Rumah Tawa Syariah,hotel,0.466942,80.665135,0.914588,0.88,0.500000,0.963185,0,Belum tersedia
6,LODGE-01018,Reddoorz Near Trans Studio Mall 2,apartment,1.648753,80.560639,0.752021,0.84,0.664709,0.996040,94,Positif
7,LODGE-01081,Vio Hotel Veteran,hotel,0.504978,79.927298,0.908269,0.86,0.500000,0.952800,0,Belum tersedia
8,LODGE-01221,Apartement Grand Asia Afrika Bandung by House ...,apartment,0.750222,79.741202,0.869532,1.00,0.500000,0.923587,0,Belum tersedia
9,LODGE-01251,Good Deal Studio Apartment At Green Kosambi Ba...,apartment,0.560928,79.675919,0.899130,1.00,0.500000,0.768339,0,Belum tersedia


,hotel_recommendation_score,distance_km_reference
count,100.000000,100.000000
mean,75.760550,1.198415
std,2.680095,0.580309
min,72.601969,0.188969
25%,73.873017,0.782321
50%,75.172311,1.160031
75%,76.804329,1.404851
max,86.330227,3.844789


Keputusan: bobot jarak dibuat 45%, paling besar dibanding rating dan sentiment. Ini cocok untuk hotel sebagai pendukung wisata terdekat.

### H6 - Endpoint Baseline Hotel

Tujuan: pastikan endpoint baseline hotel sudah tersedia di backend.

In [6]:
app_path = ROOT / "Scripts" / "app.py"
recommender_path = ROOT / "Scripts" / "penginapan_recommender.py"
app_text = app_path.read_text(encoding="utf-8")

endpoint_audit = {
    "penginapan_recommender_exists": recommender_path.exists(),
    "endpoint_registered": "/api/penginapan/recommend" in app_text,
    "schema_version_registered": "muterbandung.api.penginapan.recommend.v1" in app_text,
    "ranking_mode": "distance_weighted_baseline",
}
print(json.dumps(endpoint_audit, indent=2, ensure_ascii=False))

{
  "penginapan_recommender_exists": true,
  "endpoint_registered": true,
  "schema_version_registered": true,
  "ranking_mode": "distance_weighted_baseline"
}


Keputusan: endpoint baseline hotel tersedia. Statusnya masih baseline karena sentiment baru dari batch test.

### H7 - Output Fase Ini

Tujuan: catat file yang dihasilkan untuk langkah berikutnya.

In [7]:
output_files = [
    CURATED_DIR / "PENGINAPAN_REVIEW_SENTIMENT_INFERENCE_2026-06-10.csv",
    CURATED_DIR / "PENGINAPAN_SENTIMENT_AGGREGATED_2026-06-10.csv",
    CURATED_DIR / "PENGINAPAN_PARENT_MASTER_WITH_SENTIMENT_2026-06-10.csv",
    CURATED_DIR / "PENGINAPAN_DISTANCE_WEIGHTED_BASELINE_SAMPLE_2026-06-10.csv",
    CURATED_DIR / "penginapan_sentiment_baseline_summary_2026-06-10.json",
]

pd.DataFrame([
    {
        "file": str(path.relative_to(ROOT)),
        "exists": path.exists(),
        "size_mb": round(path.stat().st_size / 1024 / 1024, 2) if path.exists() else None,
    }
    for path in output_files
])

,file,exists,size_mb
0,Penginapan_Workspace\02_Curated\PENGINAPAN_REV...,True,0.28
1,Penginapan_Workspace\02_Curated\PENGINAPAN_SEN...,True,0.01
2,Penginapan_Workspace\02_Curated\PENGINAPAN_PAR...,True,1.79
3,Penginapan_Workspace\02_Curated\PENGINAPAN_DIS...,True,0.02
4,Penginapan_Workspace\02_Curated\penginapan_sen...,True,0.00


Ringkasan: sentiment hotel sudah bisa dipakai untuk batch awal. Langkah berikutnya adalah menjalankan proses yang sama pada full review, lalu membuat endpoint hotel baseline.

## Fase I - Review Drive dan Sentiment Hotel

Tujuan: memasukkan batch review dari zip Drive, lalu cek apakah cukup kuat untuk sentiment hotel.

In [ ]:
from pathlib import Path
import pandas as pd

zip_path = Path("drive-download-20260610T103243Z-3-001.zip")
raw_dir = Path("Penginapan_Workspace/01_Raw_Data/google_maps_reviews_json/drive_download_20260610T103243Z_3_001")
files = sorted(raw_dir.glob("*.csv"))

rows = 0
text_rows = 0
for path in files:
    df = pd.read_csv(path, usecols=lambda c: c in {"text", "textTranslated", "searchString", "title"})
    rows += len(df)
    text_rows += (
        df.get("text", pd.Series("", index=df.index)).fillna("").astype(str).str.strip().ne("")
        | df.get("textTranslated", pd.Series("", index=df.index)).fillna("").astype(str).str.strip().ne("")
    ).sum()

print("zip_exists:", zip_path.exists())
print("raw_dir:", raw_dir)
print("csv_file_count:", len(files))
print("raw_review_rows:", rows)
print("raw_rows_with_text:", int(text_rows))
print("first_files:", [p.name for p in files[:3]])

zip_exists: True
raw_dir: Penginapan_Workspace\01_Raw_Data\google_maps_reviews_json\drive_download_20260610T103243Z_3_001
csv_file_count: 12
raw_review_rows: 20597
raw_rows_with_text: 13434
first_files: ['dataset_Google-Maps-Reviews-Scraper_001.csv', 'dataset_Google-Maps-Reviews-Scraper_002.csv', 'dataset_Google-Maps-Reviews-Scraper_003.csv']


Keputusan: batch Drive diterima sebagai sumber review hotel. Isinya CSV Apify, bukan JSON, jadi normalizer dibuat membaca CSV juga.

In [ ]:
import json
from pathlib import Path

summary_path = Path("Penginapan_Workspace/02_Curated/penginapan_review_2026-06-10_drive_full_review_normalize_summary.json")
summary = json.loads(summary_path.read_text(encoding="utf-8"))
for key in [
    "raw_file_count",
    "raw_rows_read",
    "matched_rows_after_dedupe",
    "unique_review_targets_matched",
    "unique_penginapan_matched",
    "rows_with_text",
    "target_total_final_ready",
]:
    print(f"{key}: {summary.get(key)}")
print("normalized_output:", Path(summary["outputs"]["normalized"]).name)

raw_file_count: 12
raw_rows_read: 20597
matched_rows_after_dedupe: 20597
unique_review_targets_matched: 618
unique_penginapan_matched: 618
rows_with_text: 13434
target_total_final_ready: 1140
normalized_output: PENGINAPAN_REVIEW_2026-06-10_DRIVE_FULL_REVIEW_NORMALIZED.csv


Keputusan: normalisasi valid. Semua row berhasil match ke target, tapi coverage belum penuh karena target yang kena review baru sebagian.

In [ ]:
import json
from pathlib import Path

coverage_path = Path("Penginapan_Workspace/02_Curated/penginapan_review_2026-06-10_drive_full_review_coverage_summary.json")
coverage = json.loads(coverage_path.read_text(encoding="utf-8"))
for key in [
    "target_total",
    "review_rows",
    "target_with_any_scrape",
    "target_with_text_review",
    "target_missing_from_batch",
]:
    print(f"{key}: {coverage.get(key)}")
print("status_counts:", coverage.get("review_batch_status_counts"))
print("missing_file:", Path(coverage["outputs"]["missing_targets"]).name)

target_total: 1140
review_rows: 20597
target_with_any_scrape: 618
target_with_text_review: 609
target_missing_from_batch: 522
status_counts: {'scraped_with_text': 609, 'missing_from_review_batch': 522, 'scraped_star_only': 9}
missing_file: PENGINAPAN_REVIEW_2026-06-10_DRIVE_FULL_REVIEW_MISSING_TARGETS.csv


Keputusan: target yang sudah punya teks dipakai untuk sentiment. Target yang belum ada disimpan sebagai queue scraping berikutnya.

In [ ]:
import json
import pandas as pd
from pathlib import Path

summary_path = Path("Penginapan_Workspace/02_Curated/penginapan_sentiment_baseline_summary_2026-06-10_DRIVE_FULL_REVIEW.json")
summary = json.loads(summary_path.read_text(encoding="utf-8"))
for key in [
    "review_rows_input",
    "review_rows_with_text_inference",
    "parent_hotels_with_sentiment",
    "parent_master_rows",
    "parent_master_with_sentiment_rows",
    "sentiment_global_average",
    "model_warning_count",
]:
    print(f"{key}: {summary.get(key)}")

agg = pd.read_csv("Penginapan_Workspace/02_Curated/PENGINAPAN_SENTIMENT_AGGREGATED_2026-06-10_DRIVE_FULL_REVIEW.csv")
cols = [
    "parent_penginapan_id",
    "hotel_review_count_analyzed",
    "hotel_adjusted_sentiment_score",
    "hotel_review_confidence_label",
]
print(agg.sort_values("hotel_review_count_analyzed", ascending=False)[cols].head(8).to_string(index=False))

review_rows_input: 20597
review_rows_with_text_inference: 13051
parent_hotels_with_sentiment: 597
parent_master_rows: 1602
parent_master_with_sentiment_rows: 597
sentiment_global_average: 0.58946
model_warning_count: 4
parent_penginapan_id  hotel_review_count_analyzed  hotel_adjusted_sentiment_score hotel_review_confidence_label
         LODGE-00293                          318                        0.692719        high_review_confidence
         LODGE-00428                          309                        0.509729        high_review_confidence
         LODGE-01533                          259                        0.664238        high_review_confidence
         LODGE-00751                          168                        0.810457        high_review_confidence
         LODGE-01284                          166                        0.350598        high_review_confidence
         LODGE-00704                          155                        0.647712        high_review_confiden

Keputusan: sentiment hotel naik jauh. Model dipakai sebagai baseline produksi awal, dengan catatan warning versi scikit-learn perlu dibereskan saat deployment.

In [ ]:
from Scripts.penginapan_recommender import PenginapanRecommender

dataset = "Penginapan_Workspace/02_Curated/PENGINAPAN_PARENT_MASTER_WITH_SENTIMENT_2026-06-10_DRIVE_FULL_REVIEW.csv"
engine = PenginapanRecommender(dataset)
result = engine.recommend(
    query="hotel dekat alun alun bandung murah",
    user_lat=-6.9219,
    user_lon=107.6069,
    top_k=5,
)
print("total_results:", result["total_results"])
print("dataset:", result["metadata"]["dataset_path"].split("/")[-1].split("\\")[-1])
for item in result["recommendations"]:
    print(item["rank"], item["name"], item["distance_km"], item["final_score"], item["hotel_adjusted_sentiment_label"])

total_results: 487
dataset: PENGINAPAN_PARENT_MASTER_WITH_SENTIMENT_2026-06-10_DRIVE_FULL_REVIEW.csv
1 Bobobox Pods Alun Alun 0.29 99.71 Positif
2 Vasaka Maison Bandung 0.14 98.12 Positif
3 Zodiak Asia Afrika Hotel Bandung 0.24 95.75 Positif
4 Hotel Savoy Homann Bandung 0.39 95.68 Positif
5 de Braga by ARTOTEL 0.39 95.14 Positif


Keputusan: dataset baru dipakai untuk baseline recommender hotel. Bobot jarak tetap paling besar agar hotel dekat wisata/titik user naik lebih dulu.

## Fase J - Apartment Sebagai Secondary Option

Tujuan: apartment tetap disimpan, tapi tidak menjadi pilihan utama untuk query umum.

In [ ]:
import pandas as pd

path = "Penginapan_Workspace/02_Curated/PENGINAPAN_PARENT_MASTER_WITH_SENTIMENT_2026-06-10_DRIVE_FULL_REVIEW.csv"
df = pd.read_csv(path)
counts = df["property_type"].value_counts()
print("total_rows:", len(df))
print(counts.to_string())
print("apartment_share:", round(counts.get("apartment", 0) / len(df) * 100, 2), "%")

total_rows: 1602
property_type
hotel              487
apartment          352
villa              342
vacation_rental    269
guest_house        152
apartment_share: 21.97 %


Keputusan: apartment cukup besar, jadi tidak dihapus. Apartment diturunkan sebagai opsi kedua supaya rekomendasi umum tetap terasa seperti penginapan utama.

In [ ]:
from Scripts.penginapan_recommender import PenginapanRecommender

engine = PenginapanRecommender()
tests = [
    ("query_umum", "murah dekat alun alun bandung"),
    ("query_apartment", "apartemen studio murah dekat alun alun bandung"),
]

for label, query in tests:
    result = engine.recommend(query=query, user_lat=-6.9219, user_lon=107.6069, top_k=5)
    print("\n" + label + ":", query)
    for item in result["recommendations"]:
        print(
            item["rank"],
            item["property_type"],
            item["name"],
            item["final_score"],
            item["score_breakdown"]["property_type_priority_score"],
        )

query_umum: murah dekat alun alun bandung
1 hotel Bobobox Pods Alun Alun 95.87 1.0
2 hotel Vasaka Maison Bandung 94.28 1.0
3 hotel Zodiak Asia Afrika Hotel Bandung 91.67 1.0
4 hotel Hotel Savoy Homann Bandung 91.6 1.0
5 hotel de Braga by ARTOTEL 91.06 1.0

query_apartment: apartemen studio murah dekat alun alun bandung
1 apartment High Livin Apartment Asia Afrika 83.75 1.0
2 apartment Platinum 1010 Studio Tera Apartemen Bandung City View - One-Bedroom Apartment 83.22 1.0
3 apartment Homey 3Br Apartment At Braga City Walk 82.86 1.0
4 apartment Best Location 2Br At Braga City Walk Apartment 82.59 1.0
5 apartment Platinum 1012 Studio Tera Apartemen Bandung City View 82.15 1.0


Keputusan: query umum tidak didominasi apartment. Kalau user memang mencari apartment/studio, penalti dilepas dan apartment tampil normal.